In [2]:
import sys
import os
from pathlib import Path

# Set the working directory to project root
project_root = Path().resolve().parents[0]

# Add 'src' directory
sys.path.append(str(project_root / "src"))
sys.path.append(str(project_root / "data"))
print(sys.path)

['/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys', '/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev', '/Applications/PyCharm.app/Contents/plugins/python/helpers-pro/jupyter_debug', '/opt/homebrew/Cellar/python@3.11/3.11.11/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/opt/homebrew/Cellar/python@3.11/3.11.11/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/opt/homebrew/Cellar/python@3.11/3.11.11/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/.venv/lib/python3.11/site-packages', '/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/.venv/lib/python3.11/site-packages/setuptools/_vendor', '/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/notebooks/src', '/Users/go82gax/Documents/Projekte/LFS/Analys

In [3]:
# import packages and modules

import numpy as np
import pandas as pd

from data.framework import Esco, Classifications
from data.lfs import EuLfs
from src import utils

# Load central paths object
useful_paths = utils.UsefulPaths(fn_config_path="paths_config.yml")
lfs_config = utils.load_config(os.path.join(useful_paths.config_dir, "eu_lfs_config.yml"))

In [6]:
import pickle

# --- digital skills ---
digital_csv = os.path.join(
    useful_paths.data_raw,
    "esco",
    "v1.1.0",
    "digCompSkillsCollection_en.csv",
)
df_dig = pd.read_csv(digital_csv, index_col=0)
dig_skills = set(df_dig["preferredLabel"])
print("Digital skills total:", len(dig_skills))

# --- green skills ---
green_csv = os.path.join(
    useful_paths.data_raw,
    "esco",
    "v1.1.0",
    "greenSkillsCollection_en.csv",
)
df_green = pd.read_csv(green_csv, index_col=0)
green_skills = set(df_green["preferredLabel"])
print("Green   skills total:", len(green_skills))

# --- occupation-skills matrix ---
occ_pkl = os.path.join(
    useful_paths.data_processed,
    "esco",
    "occ_skills_matrix.pkl",
)

with open(occ_pkl, "rb") as f:
    occ_skills_mat_3d = pickle.load(f)
matrix_skills = set(occ_skills_mat_3d.columns)
print("Matrix skill‐columns:", len(matrix_skills))

Digital skills total: 21
Green   skills total: 570
Matrix skill‐columns: 13891


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_99870/2906702465.py:33: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills_mat_3d = pickle.load(f)


In [7]:
# --- compute overlaps ---
dig_overlap   = dig_skills   & matrix_skills
green_overlap = green_skills & matrix_skills

print(f"Digital overlap: {len(dig_overlap)} / {len(dig_skills)} "
      f"({100 * len(dig_overlap)/len(dig_skills):.1f}%)")
print(f"Green   overlap: {len(green_overlap)} / {len(green_skills)} "
      f"({100 * len(green_overlap)/len(green_skills):.1f}%)")

Digital overlap: 0 / 21 (0.0%)
Green   overlap: 0 / 570 (0.0%)


In [11]:
import os
import pandas as pd
import pickle

# 1) Load the coreness DataFrame
path_coreness = os.path.join(useful_paths.data_processed, "esco", "skills_network_metrics.pkl")
df_coreness = pd.read_pickle(path_coreness)
coreness_labels = set(df_coreness["preferredLabel"])
print("Coreness labels total:", len(coreness_labels))

# 2) Load the digital and green CSVs
digital_csv = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "digCompSkillsCollection_en.csv")
green_csv   = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv")

df_dig   = pd.read_csv(digital_csv, index_col=0)
df_green = pd.read_csv(green_csv,   index_col=0)

digital_skills = set(df_dig["preferredLabel"])
green_skills   = set(df_green["preferredLabel"])
print("Digital skills total:", len(digital_skills))
print("Green   skills total:",   len(green_skills))

# 3) Compute overlap
dig_overlap   = digital_skills   & coreness_labels
green_overlap = green_skills     & coreness_labels
print(f"Digital overlap: {len(dig_overlap)} / {len(digital_skills)} ({100*len(dig_overlap)/len(digital_skills):.1f}%)")
print(f"Green   overlap: {len(green_overlap)} / {len(green_skills)}   ({100*len(green_overlap)/len(green_skills):.1f}%)")

# if you want to inspect which ones are missing:
missing_dig   = sorted(digital_skills   - coreness_labels)
missing_green = sorted(green_skills     - coreness_labels)
pd.Series(missing_dig).to_csv("missing_digital_labels.csv", index=False)
pd.Series(missing_green).to_csv("missing_green_labels.csv", index=False)


Coreness labels total: 13891
Digital skills total: 21
Green   skills total: 570
Digital overlap: 21 / 21 (100.0%)
Green   overlap: 570 / 570   (100.0%)


In [13]:
# quick inspection
print("df_coreness columns:", df_coreness.columns.tolist())
print(df_coreness.iloc[0])

df_coreness columns: ['conceptType', 'conceptUri', 'skillType', 'reuseLevel', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'scopeNote', 'definition', 'inScheme', 'description', 'eigenvector_centrality', 'betweenness_centrality', 'betweenness_centrality_norm', 'mean_centrality', 'clustering_coefficient', 'coreness', 'skill_classification_esco']
conceptType                                             KnowledgeSkillCompetence
conceptUri                     http://data.europa.eu/esco/skill/0005c151-5b5a...
skillType                                                       skill/competence
reuseLevel                                                       sector-specific
preferredLabel                                              manage musical staff
altLabels                      manage staff of music\ncoordinate duties of mu...
hiddenLabels                                                                 NaN
status                                                     

In [14]:
chosen = next(iter(dig_skills))  # or any digital label
row = df_coreness[df_coreness["preferredLabel"] == chosen].iloc[0]
uri = row["conceptUri"]
pos = list(occ_skills_mat_3d.columns).index(uri)
print(chosen, "→", uri, "@", pos)

copyright and licenses related to digital content → http://data.europa.eu/esco/skill/a91732ce-988e-4105-9570-f425c6ffdc82 @ 9215


In [15]:
import os
import pandas as pd

# 1) Load the coreness DataFrame
path_coreness = os.path.join(useful_paths.data_processed, "esco", "skills_network_metrics.pkl")
df_coreness = pd.read_pickle(path_coreness)
coreness_labels = set(df_coreness["preferredLabel"])
print("Coreness labels total:", len(coreness_labels))

# 2) Load the digital and green CSVs
digital_csv = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "digCompSkillsCollection_en.csv")
green_csv   = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv")

df_dig   = pd.read_csv(digital_csv, index_col=0)
df_green = pd.read_csv(green_csv,   index_col=0)

digital_skills = set(df_dig["preferredLabel"])
green_skills   = set(df_green["preferredLabel"])
print("Digital skills total:", len(digital_skills))
print("Green   skills total:",   len(green_skills))

# 3) Compute overlap with your coreness set
dig_overlap   = digital_skills   & coreness_labels
green_overlap = green_skills     & coreness_labels
print(f"Digital overlap: {len(dig_overlap)} / {len(digital_skills)} ({100*len(dig_overlap)/len(digital_skills):.1f}%)")
print(f"Green   overlap: {len(green_overlap)} / {len(green_skills)}   ({100*len(green_overlap)/len(green_skills):.1f}%)")

# 4) (Optional) write out missing ones for inspection
missing_dig   = sorted(digital_skills   - coreness_labels)
missing_green = sorted(green_skills     - coreness_labels)
pd.Series(missing_dig).to_csv("missing_digital_labels.csv", index=False)
pd.Series(missing_green).to_csv("missing_green_labels.csv", index=False)


Coreness labels total: 13891
Digital skills total: 21
Green   skills total: 570
Digital overlap: 21 / 21 (100.0%)
Green   overlap: 570 / 570   (100.0%)


In [18]:
# 1) Load your saved occ–skills matrix
path_mat = os.path.join(useful_paths.data_processed, "esco", "occ_skills_matrix.pkl")
occ_skills_mat = pd.read_pickle(path_mat)
# if that is the raw bipartite (multi-index) version, you might need:
# occ_skills_mat.index = pd.MultiIndex.from_tuples(occ_skills_mat.index)
# occ_skills_mat_3d = occ_skills_mat.groupby(level=3).mean()
# else, if you have already a 3d matrix, skip the above and just:
occ_skills_mat_3d = occ_skills_mat

# 2) Peek at its columns
matrix_cols = list(occ_skills_mat_3d.columns)
print("First 10 matrix columns:\n", matrix_cols[:10])

# 3) Load your digital & green CSVs just as in __init__
df_dig   = pd.read_csv(
    os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "digCompSkillsCollection_en.csv"),
    index_col=0,
)
df_green = pd.read_csv(
    os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv"),
    index_col=0,
)

all_dig_uris   = df_dig["conceptUri"].tolist()
all_green_uris = df_green["conceptUri"].tolist()

# 4) Exact‐URI match test
present_dig_exact = [u for u in all_dig_uris   if u in matrix_cols]
present_grn_exact = [u for u in all_green_uris if u in matrix_cols]
print(f"\nDigital exact matches: {len(present_dig_exact)}/{len(all_dig_uris)}")
print(f"Green   exact matches: {len(present_grn_exact)}/{len(all_green_uris)}")

# 5) Tail‐segment match test
matrix_tails = {c.split("/")[-1] for c in matrix_cols}
dig_tails    = [u.split("/")[-1] for u in all_dig_uris]
grn_tails    = [u.split("/")[-1] for u in all_green_uris]

present_dig_tail = [t for t in dig_tails if t in matrix_tails]
present_grn_tail = [t for t in grn_tails if t in matrix_tails]
print(f"\nDigital tail‐matches: {len(present_dig_tail)}/{len(dig_tails)}")
print(f"Green   tail‐matches: {len(present_grn_tail)}/{len(grn_tails)}")

First 10 matrix columns:
 ['http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab', 'http://data.europa.eu/esco/skill/00064735-8fad-454b-90c7-ed858cc993f2', 'http://data.europa.eu/esco/skill/000709ed-2be5-4193-b056-45a97698d828', 'http://data.europa.eu/esco/skill/0007bdc2-dd15-4824-b7d6-416522c46f35', 'http://data.europa.eu/esco/skill/00090cc1-1f27-439e-a4e0-19a87a501bfc', 'http://data.europa.eu/esco/skill/000bb1e4-89f0-4b86-be05-05ece3641724', 'http://data.europa.eu/esco/skill/000c94d2-2a2e-4545-993c-6df8cb5b0316', 'http://data.europa.eu/esco/skill/000f1d3d-220f-4789-9c0a-cc742521fb02', 'http://data.europa.eu/esco/skill/001115fb-569f-4ee6-8381-c6807ef2527f', 'http://data.europa.eu/esco/skill/001d46db-035e-4b92-83a3-ed8771e0c123']

Digital exact matches: 21/21
Green   exact matches: 570/570

Digital tail‐matches: 21/21
Green   tail‐matches: 570/570


In [21]:
import os
import pickle
import numpy as np
import pandas as pd

# adjust these paths to match your setup
from pathlib import Path
base = Path(useful_paths.data_processed) / "esco"
matrix_path = base / "occ_skills_matrix.pkl"

# 1) Load the occ–skills matrix (3d version)
with open(matrix_path, "rb") as f:
    mat = pickle.load(f)    # pandas DataFrame, shape ~(n_occ × n_skill)
print("Matrix shape:", mat.shape)
print("First 5 skill URIs:", mat.columns[:5].tolist())
print("First 5 occ URIs:  ", mat.index[:5].tolist())

# 2) Load your digital‐skills list from __init__
dig_csv = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "digCompSkillsCollection_en.csv"
df_dig = pd.read_csv(dig_csv, index_col=0)
digital_uris = df_dig["conceptUri"].tolist()
print(f"Total digital URIs: {len(digital_uris)}")
print("Found in matrix:", sum(u in mat.columns for u in digital_uris))

# 3) Pick one test occupation URI and get its row index
occ_uri = mat.index[0]   # ← you can choose any other by position or full URI
print("\nTesting occupation URI:", occ_uri)
occ_idx = mat.index.get_loc(occ_uri)

# 4) Baseline similarity vector
base_sim = mat.values[occ_idx] @ mat.values.T
base_sim[occ_idx] = 0
print("\nBaseline similarity:")
print("  max         =", base_sim.max())
print("  90th percentile =", np.percentile(base_sim, 90))

# 5) Now add up to N digital skills one by one, cumulatively
print("\nAdding skills cumulatively:")
have = set(mat.columns[mat.values[occ_idx] > 0])
remaining = [u for u in digital_uris if u not in have]

# for reproducibility
rng = np.random.default_rng(0)

for step in range(1, min(21, len(remaining)) + 1):
    # pick one at random from those not yet added
    pick = rng.choice(remaining, replace=False)
    remaining.remove(pick)
    # update a copy of the row’s skill vector
    vec = mat.values[occ_idx].copy()
    skill_j = mat.columns.get_loc(pick)
    vec[skill_j] = 1
    # compute new similarity
    sim = vec @ mat.values.T
    sim[occ_idx] = 0
    print(f" Step {step:2d}: added {pick.split('/')[-1]} → max sim = {sim.max():.3f}")

# If sim.max() never exceeds your viability threshold (e.g. 3.7),
# then there simply aren’t enough overlaps under the raw dot‐product metric.


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_99870/1235843361.py:13: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  mat = pickle.load(f)    # pandas DataFrame, shape ~(n_occ × n_skill)


Matrix shape: (3008, 13891)
First 5 skill URIs: ['http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab', 'http://data.europa.eu/esco/skill/00064735-8fad-454b-90c7-ed858cc993f2', 'http://data.europa.eu/esco/skill/000709ed-2be5-4193-b056-45a97698d828', 'http://data.europa.eu/esco/skill/0007bdc2-dd15-4824-b7d6-416522c46f35', 'http://data.europa.eu/esco/skill/00090cc1-1f27-439e-a4e0-19a87a501bfc']
First 5 occ URIs:   ['http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34', 'http://data.europa.eu/esco/occupation/000e93a3-d956-4e45-aacb-f12c83fedf84', 'http://data.europa.eu/esco/occupation/0019b951-c699-4191-8208-9822882d150c', 'http://data.europa.eu/esco/occupation/0022f466-426c-41a4-ac96-a235c945cf97', 'http://data.europa.eu/esco/occupation/002da35b-7808-43f3-83bf-63596b8b351f']
Total digital URIs: 21
Found in matrix: 21

Testing occupation URI: http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34

Baseline similarity:
  max     

In [22]:
import os
import pickle
import numpy as np
import pandas as pd

# 1) Load your 3 008×13 891 occ–skill matrix
mat_path = os.path.join(useful_paths.data_processed, "esco", "occ_skills_matrix.pkl")
with open(mat_path, "rb") as f:
    mat: pd.DataFrame = pickle.load(f)

print(f"Matrix shape: {mat.shape}")
print("First 5 skill URIs:", list(mat.columns[:5]))
print("First 5 occ URIs:  ", list(mat.index[:5]))

# 2) Load your green list (or use self.green_skills if you already have it)
green_csv = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv")
df_green = pd.read_csv(green_csv, index_col=0)
green_uris = df_green["conceptUri"].tolist()
print(f"Total green URIs: {len(green_uris)}")
print(f"Found in matrix: {sum(uri in mat.columns for uri in green_uris)}")

# 3) Pick a test occupation URI (here the first one in the matrix)
test_occ = mat.index[0]
idx_occ = mat.index.get_loc(test_occ)
print("\nTesting occupation URI:", test_occ)

# 4) Baseline similarity (dot-product)
baseline = mat.values[idx_occ] @ mat.values.T
print("\nBaseline max similarity:", baseline.max())

# 5) Cumulatively add each green skill and recompute
print("\nAdding green skills cumulatively:")
for step, uri in enumerate(green_uris, start=1):
    X = mat.copy()
    skill_idx = mat.columns.get_loc(uri)
    X.values[idx_occ, skill_idx] = 1
    sims = X.values[idx_occ] @ X.values.T
    print(f" Step {step:3d}: added {uri.split('/')[-1]} → max sim = {sims.max():.3f}")


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_99870/407426595.py:9: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  mat: pd.DataFrame = pickle.load(f)


Matrix shape: (3008, 13891)
First 5 skill URIs: ['http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab', 'http://data.europa.eu/esco/skill/00064735-8fad-454b-90c7-ed858cc993f2', 'http://data.europa.eu/esco/skill/000709ed-2be5-4193-b056-45a97698d828', 'http://data.europa.eu/esco/skill/0007bdc2-dd15-4824-b7d6-416522c46f35', 'http://data.europa.eu/esco/skill/00090cc1-1f27-439e-a4e0-19a87a501bfc']
First 5 occ URIs:   ['http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34', 'http://data.europa.eu/esco/occupation/000e93a3-d956-4e45-aacb-f12c83fedf84', 'http://data.europa.eu/esco/occupation/0019b951-c699-4191-8208-9822882d150c', 'http://data.europa.eu/esco/occupation/0022f466-426c-41a4-ac96-a235c945cf97', 'http://data.europa.eu/esco/occupation/002da35b-7808-43f3-83bf-63596b8b351f']
Total green URIs: 570
Found in matrix: 570

Testing occupation URI: http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34

Baseline max similarity: 8

Add

In [24]:
import os
import pickle
import pandas as pd

# 1) load the occ–skills matrix (essential only, binary)
with open(os.path.join(useful_paths.data_processed, "esco", "occ_skills_matrix.pkl"), "rb") as f:
    occ_skills = pickle.load(f)  # DataFrame: rows=occupation URIs, cols=skill URIs

# 2) load the digital and green lists (with conceptUri column)
df_dig = pd.read_csv(
    os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "digCompSkillsCollection_en.csv"),
    index_col=0,
)
df_grn = pd.read_csv(
    os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv"),
    index_col=0,
)
digital_uris = df_dig["conceptUri"].tolist()
green_uris   = df_grn["conceptUri"].tolist()

# 3) count, for each skill, how many occupations list it as essential
dig_counts = [(uri, (occ_skills.get(uri, 0) == 1).sum()) for uri in digital_uris]
grn_counts = [(uri, (occ_skills.get(uri, 0) == 1).sum()) for uri in green_uris]

# 4) summary
print("Digital skills ever essential:", sum(1 for _, c in dig_counts if c>0), "/", len(dig_counts))
print("Green   skills ever essential:", sum(1 for _, c in grn_counts if c>0), "/", len(grn_counts))

# 5) optional check: how many occupations per skill
print("\nDigital skill counts (occupation hits):")
print(pd.Series(dict(dig_counts)).describe().to_frame().T)
print("\nGreen skill counts (occupation hits):")
print(pd.Series(dict(grn_counts)).describe().to_frame().T)

/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_99870/3955065981.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)  # DataFrame: rows=occupation URIs, cols=skill URIs


Digital skills ever essential: 5 / 21
Green   skills ever essential: 475 / 570

Digital skill counts (occupation hits):
   count      mean       std  min  25%  50%  75%   max
0   21.0  1.428571  4.801785  0.0  0.0  0.0  0.0  22.0

Green skill counts (occupation hits):
   count      mean       std  min  25%  50%  75%    max
0  570.0  4.268421  8.528246  0.0  1.0  2.0  4.0  108.0


In [25]:
# 6) Which digital skills are ever essential?
dig_essentials = [uri for uri, count in dig_counts if count > 0]
print("Digital skills ever essential (URIs):")
for uri in dig_essentials:
    print(f"  {uri}  (essential in {(occ_skills[uri] == 1).sum()} occupations)")

# 7) Which green skill is required by the most occupations?
top_green_uri, top_green_count = max(grn_counts, key=lambda x: x[1])
print(f"\nMost‐ubiquitous green skill:\n  {top_green_uri}  (essential in {top_green_count} occupations)")

# 8) Which occupations require the most green skills?
#    Count per occupation how many green skills they list as essential
occ_green_counts = occ_skills[green_uris].sum(axis=1)
top5_occs = occ_green_counts.nlargest(5)
print("\nTop 5 occupations by number of essential green skills:")
for occ_uri, n in top5_occs.items():
    print(f"  {occ_uri}  ({int(n)} green skills)")

Digital skills ever essential (URIs):
  http://data.europa.eu/esco/skill/14832d87-2f2f-4895-b290-e4760ebae42a  (essential in 3 occupations)
  http://data.europa.eu/esco/skill/21d2f96d-35f7-4e3f-9745-c533d2dd6e97  (essential in 22 occupations)
  http://data.europa.eu/esco/skill/2b34a99f-9813-4c91-9509-b6b9b8c3132e  (essential in 1 occupations)
  http://data.europa.eu/esco/skill/33a82b83-c838-4889-ae62-fae1317481eb  (essential in 3 occupations)
  http://data.europa.eu/esco/skill/4d97e3c3-f335-47cc-a4ee-0d779fd42222  (essential in 1 occupations)

Most‐ubiquitous green skill:
  http://data.europa.eu/esco/skill/86df7af2-f9f3-4c06-a500-7f1fba9e78fe  (essential in 108 occupations)

Top 5 occupations by number of essential green skills:
  http://data.europa.eu/esco/occupation/b2cede50-82bb-4684-9f11-1930e12ad672  (179 green skills)
  http://data.europa.eu/esco/occupation/d7d986e1-7333-431b-9719-0c5c6939e360  (145 green skills)
  http://data.europa.eu/esco/occupation/579254cf-6d69-4889-9000-9c7

In [31]:
# — after your dig_counts / grn_counts code block —
from data.framework import Esco
esco = Esco()

# ————————————————————————————————
# build URI → preferredLabel maps
# ————————————————————————————————
skill_labels = df_coreness.set_index("conceptUri")["preferredLabel"]
occ_labels   = esco.occupations.set_index("conceptUri")["preferredLabel"]

# ————————————————————————————————
# which digital skills ever essential?
# (dig_counts is [(uri, count), …] from your code)
# ————————————————————————————————
dig_essentials = [uri for uri, c in dig_counts if c > 0]

# ————————————————————————————————
# which green skill is most ubiquitous?
# ————————————————————————————————
top_green_uri = max(grn_counts, key=lambda x: x[1])[0]

# ————————————————————————————————
# and which occupations have the most essential green skills?
# first compute per-occupation green count:
grn_by_occ = occ_skills[green_uris].sum(axis=1)   # Series indexed by occupation URI
top5_occs  = grn_by_occ.nlargest(5)

# ————————————————————————————————
# now print them, with labels
# ————————————————————————————————
print("Digital skills ever essential:")
for uri in dig_essentials:
    print(f" • {skill_labels[uri]}  ({uri})")

print(f"\nMost‐ubiquitous green skill:\n • {skill_labels[top_green_uri]}  ({top_green_uri})")

print("\nTop 5 occupations by essential green-skill count:")
for uri, n in top5_occs.items():
    print(f" • {occ_labels[uri]}  ({uri}) — {n} green skills")

Digital skills ever essential:
 • solve technical problems  (http://data.europa.eu/esco/skill/14832d87-2f2f-4895-b290-e4760ebae42a)
 • computer programming  (http://data.europa.eu/esco/skill/21d2f96d-35f7-4e3f-9745-c533d2dd6e97)
 • collaborate through digital technologies  (http://data.europa.eu/esco/skill/2b34a99f-9813-4c91-9509-b6b9b8c3132e)
 • protect personal data and privacy  (http://data.europa.eu/esco/skill/33a82b83-c838-4889-ae62-fae1317481eb)
 • manage data, information and digital content  (http://data.europa.eu/esco/skill/4d97e3c3-f335-47cc-a4ee-0d779fd42222)

Most‐ubiquitous green skill:
 • follow health and safety procedures in construction  (http://data.europa.eu/esco/skill/86df7af2-f9f3-4c06-a500-7f1fba9e78fe)

Top 5 occupations by essential green-skill count:
 • energy engineer  (http://data.europa.eu/esco/occupation/b2cede50-82bb-4684-9f11-1930e12ad672) — 179 green skills
 • civil engineer  (http://data.europa.eu/esco/occupation/d7d986e1-7333-431b-9719-0c5c6939e360) — 

In [35]:
from data.framework import Esco
esco = Esco()

# build URI → preferredLabel maps
skill_labels = df_coreness.set_index("conceptUri")["preferredLabel"]
occ_labels   = esco.occupations.set_index("conceptUri")["preferredLabel"]

# which digital skills ever essential?
dig_essentials = [uri for uri, c in dig_counts if c > 0]

# which green skill is most ubiquitous?
top_green_uri = max(grn_counts, key=lambda x: x[1])[0]

# Top 5 occupations by essential green‐skill count
grn_by_occ    = occ_skills[green_uris].sum(axis=1).astype(int)
top5_grn_occs = grn_by_occ.nlargest(5)

# Top 5 occupations by essential digital‐skill count
dig_by_occ    = occ_skills[digital_uris].sum(axis=1).astype(int)
top5_dig_occs = dig_by_occ.nlargest(5)

# print with labels
print("Digital skills ever essential:")
for uri in dig_essentials:
    print(f" • {skill_labels[uri]}  ({uri})")

print(f"\nMost‐ubiquitous green skill:\n • {skill_labels[top_green_uri]}  ({top_green_uri})")

print("\nTop 5 occupations by essential green-skill count:")
for uri, n in top5_grn_occs.items():
    print(f" • {occ_labels[uri]}  ({uri}) — {n} green skills")

print("\nTop 5 occupations by essential digital-skill count:")
for uri, n in top5_dig_occs.items():
    print(f" • {occ_labels[uri]}  ({uri}) — {n} digital skills")


Digital skills ever essential:
 • solve technical problems  (http://data.europa.eu/esco/skill/14832d87-2f2f-4895-b290-e4760ebae42a)
 • computer programming  (http://data.europa.eu/esco/skill/21d2f96d-35f7-4e3f-9745-c533d2dd6e97)
 • collaborate through digital technologies  (http://data.europa.eu/esco/skill/2b34a99f-9813-4c91-9509-b6b9b8c3132e)
 • protect personal data and privacy  (http://data.europa.eu/esco/skill/33a82b83-c838-4889-ae62-fae1317481eb)
 • manage data, information and digital content  (http://data.europa.eu/esco/skill/4d97e3c3-f335-47cc-a4ee-0d779fd42222)

Most‐ubiquitous green skill:
 • follow health and safety procedures in construction  (http://data.europa.eu/esco/skill/86df7af2-f9f3-4c06-a500-7f1fba9e78fe)

Top 5 occupations by essential green-skill count:
 • energy engineer  (http://data.europa.eu/esco/occupation/b2cede50-82bb-4684-9f11-1930e12ad672) — 179 green skills
 • civil engineer  (http://data.europa.eu/esco/occupation/d7d986e1-7333-431b-9719-0c5c6939e360) — 

In [36]:
# — after your dig_counts / grn_counts code block —
# 1) make a Series of green‐skill → essential‐occupation‐count
green_series = pd.Series({uri: (occ_skills.get(uri,0)==1).sum()
                          for uri in green_uris})
# 2) pick top 5
top5_green = green_series.nlargest(5)

# 3) map to labels
skill_labels = df_coreness.set_index("conceptUri")["preferredLabel"]
for uri, count in top5_green.items():
    print(f"{skill_labels[uri]} — {count} occupations")

follow health and safety procedures in construction — 108 occupations
corporate social responsibility — 75 occupations
ensure compliance with environmental legislation — 67 occupations
environmental legislation — 53 occupations
ensure correct goods labelling — 52 occupations


In [39]:
import os
import pickle
import pandas as pd
from data.framework import Esco

# 1) load your essential (binary) occ–skills matrix
with open(os.path.join(useful_paths.data_processed, "esco", "occ_skills_matrix.pkl"), "rb") as f:
    occ_skills: pd.DataFrame = pickle.load(f)
#    rows = occupation URIs, cols = skill URIs

# 2) load ESCO metadata so we can map occupations → iscoGroup
esco = Esco()
df_occ_meta = esco.occupations[['conceptUri','iscoGroup']].copy()
# ensure it's string and take first 3 digits of the code
df_occ_meta['isco3'] = df_occ_meta['iscoGroup'].astype(str).str[:3]

# build lookup: occ URI → isco3
occ_to_isco3 = df_occ_meta.set_index('conceptUri')['isco3']

# 3) load your digital & green URIs
df_dig = pd.read_csv(
    os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "digCompSkillsCollection_en.csv"),
    index_col=0,
)
df_grn = pd.read_csv(
    os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv"),
    index_col=0,
)
digital_uris = df_dig["conceptUri"].tolist()
green_uris   = df_grn["conceptUri"].tolist()

# 4) count per‐skill how many *distinct* ISCO-3 groups list it as essential
dig_isco3_counts = {
    uri: ( occ_to_isco3[ occ_skills.index[ occ_skills[uri]==1 ] ]
           .dropna()
           .nunique()
         )
    for uri in digital_uris
}
grn_isco3_counts = {
    uri: ( occ_to_isco3[ occ_skills.index[ occ_skills[uri]==1 ] ]
           .dropna()
           .nunique()
         )
    for uri in green_uris
}

# 5) turn into Series and summarise
dig3 = pd.Series(dig_isco3_counts)
grn3 = pd.Series(grn_isco3_counts)

print("Digital skills ever essential (distinct ISCO-3 groups):", (dig3>0).sum(), "/", len(dig3))
print("Green   skills ever essential (distinct ISCO-3 groups):", (grn3>0).sum(), "/", len(grn3))

print("\nDigital ISCO-3 coverage summary:")
print(dig3.describe().to_frame().T)

print("\nGreen ISCO-3 coverage summary:")
print(grn3.describe().to_frame().T)


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_99870/3276570572.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills: pd.DataFrame = pickle.load(f)


Digital skills ever essential (distinct ISCO-3 groups): 5 / 21
Green   skills ever essential (distinct ISCO-3 groups): 475 / 570

Digital ISCO-3 coverage summary:
   count      mean       std  min  25%  50%  75%  max
0   21.0  0.666667  1.527525  0.0  0.0  0.0  0.0  6.0

Green ISCO-3 coverage summary:
   count      mean       std  min  25%  50%  75%   max
0  570.0  2.277193  2.668637  0.0  1.0  1.5  3.0  28.0


In [41]:
# assumes 'esco' is your ESCO helper from data.framework import Esco
# and that esco.occupations has columns ['conceptUri','iscoGroup']

# 1) build mapping from each specific occ URI to its ISCO-4 group
occ_meta = esco.occupations.set_index("conceptUri")[["iscoGroup","preferredLabel"]]
uri_to_group = occ_meta["iscoGroup"].to_dict()
group_to_label = occ_meta.drop_duplicates("iscoGroup").set_index("iscoGroup")["preferredLabel"].to_dict()

# 2) collapse the occ_skills matrix from URI→group
occ_skills_group = occ_skills.copy()
# map every row index (URI) to its group code; drop any URI without a mapping
occ_skills_group.index = occ_skills_group.index.map(uri_to_group)
occ_skills_group = occ_skills_group[~occ_skills_group.index.isna()]
# now sum up all the rows belonging to the same group
occ_skills_by_group = occ_skills_group.groupby(level=0).sum()

# 3) recompute the “top 5 occupations by essential green‐skill count” at group level
green_counts_by_group = occ_skills_by_group[green_uris].sum(axis=1)
top5_green = green_counts_by_group.nlargest(5)
print("Top 5 ISCO-4 groups by essential green‐skill count:")
for grp, cnt in top5_green.items():
    print(f" • {group_to_label.get(grp,grp)} (ISCO-4 {grp}) — {cnt} green skills")

# 4) likewise for digital skills
digital_counts_by_group = occ_skills_by_group[digital_uris].sum(axis=1)
top5_dig = digital_counts_by_group.nlargest(5)
print("\nTop 5 ISCO-4 groups by essential digital‐skill count:")
for grp, cnt in top5_dig.items():
    print(f" • {group_to_label.get(grp,grp)} (ISCO-4 {grp}) — {cnt} digital skills")

Top 5 ISCO-4 groups by essential green‐skill count:
 • dismantling engineer (ISCO-4 2149) — 529 green skills
 • environmental scientist (ISCO-4 2133) — 458 green skills
 • geological engineer (ISCO-4 2142) — 295 green skills
 • steam engineer (ISCO-4 2144) — 266 green skills
 • footwear quality manager (ISCO-4 1321) — 234 green skills

Top 5 ISCO-4 groups by essential digital‐skill count:
 • general practitioner (ISCO-4 2211) — 14 digital skills
 • fiberglass laminator (ISCO-4 8142) — 14 digital skills
 • integration engineer (ISCO-4 2511) — 13 digital skills
 • specialised doctor (ISCO-4 2212) — 12 digital skills
 • land-based machinery technician (ISCO-4 7233) — 11 digital skills


In [42]:
# 2b) collapse via OR (max) instead of sum
occ_skills_group = occ_skills.copy()
occ_skills_group.index = occ_skills_group.index.map(uri_to_group)
occ_skills_group = occ_skills_group[~occ_skills_group.index.isna()]

# GROUP-OR: if any child has it, the group has it
occ_skills_by_group = (
    occ_skills_group
    .groupby(level=0)
    .max()
)

# now counts are bounded by the total number of skills
green_counts_by_group = occ_skills_by_group[green_uris].sum(axis=1)
digital_counts_by_group = occ_skills_by_group[digital_uris].sum(axis=1)

# Top 5 at ISCO-4:
print("Top 5 ISCO-4 groups by essential green-skill count (corrected):")
for grp, cnt in green_counts_by_group.nlargest(5).items():
    print(f" • {group_to_label.get(grp,grp)} (ISCO-4 {grp}) — {int(cnt)} green skills")

print("\nTop 5 ISCO-4 groups by essential digital-skill count (corrected):")
for grp, cnt in digital_counts_by_group.nlargest(5).items():
    print(f" • {group_to_label.get(grp,grp)} (ISCO-4 {grp}) — {int(cnt)} digital skills")


Top 5 ISCO-4 groups by essential green-skill count (corrected):
 • dismantling engineer (ISCO-4 2149) — 268 green skills
 • environmental scientist (ISCO-4 2133) — 260 green skills
 • geological engineer (ISCO-4 2142) — 205 green skills
 • steam engineer (ISCO-4 2144) — 161 green skills
 • footwear quality manager (ISCO-4 1321) — 138 green skills

Top 5 ISCO-4 groups by essential digital-skill count (corrected):
 • general practitioner (ISCO-4 2211) — 14 digital skills
 • specialised doctor (ISCO-4 2212) — 12 digital skills
 • gear machinist (ISCO-4 7223) — 4 digital skills
 • telecommunications manager (ISCO-4 1330) — 2 digital skills
 • clothing technologist (ISCO-4 2141) — 2 digital skills


In [4]:
# ---------- paste this at the end of your Trials notebook ----------
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import glob
import textwrap

# useful_paths should already be defined in your notebook; if not, adapt base manually:
try:
    base_esco = Path(useful_paths.data_processed) / "esco"
except Exception:
    base_esco = Path("data/processed/esco")   # fallback - adjust if your layout differs
print("Looking in:", base_esco)

# 1) list candidate occ_skills files
cand = sorted([p for p in base_esco.iterdir() if "occ_skills_matrix" in p.name])
print("\nCandidate occ_skills files found:")
for p in cand:
    print(" -", p.name)

if not cand:
    raise RuntimeError("No occ_skills_matrix files found in the expected folder. Adjust base_esco path.")

# helper to summarize a loaded matrix-like object
def summarize_matrix(obj):
    """
    Accepts: pandas DataFrame or numpy array-like.
    Returns dict with shape, dtype, min/max, percentiles, integer-check, unique-count (capped).
    """
    if isinstance(obj, pd.DataFrame):
        arr = obj.values
    else:
        arr = np.asarray(obj)
    out = {}
    out["shape"] = arr.shape
    out["dtype"] = arr.dtype
    # convert to 1d sample for expensive ops if huge
    flat = arr.ravel()
    # compute numeric summary (use percentiles to avoid huge unique costs)
    out["min"] = float(np.nanmin(flat))
    out["max"] = float(np.nanmax(flat))
    out["mean"] = float(np.nanmean(flat))
    out["p10"], out["p25"], out["p50"], out["p75"], out["p90"] = (
        float(np.nanpercentile(flat, q)) for q in (10,25,50,75,90)
    )
    # are all values 0/1?
    # note: allow float closeness (tolerance)
    is_binary = np.all(np.isin(np.unique(np.round(flat,6)), [0.0,1.0]))
    out["binary_like_0_1"] = bool(is_binary)
    # are values effectively integers?
    out["all_values_integer"] = bool(np.allclose(flat, np.round(flat), atol=1e-8))
    # how many unique values (capped)
    uniq = np.unique(flat)
    out["unique_count"] = int(uniq.size) if uniq.size < 10000 else f"{uniq.size} (capped)"
    # if not binary, sample top unique values and example values
    out["unique_sample_first10"] = uniq[:10].tolist() if uniq.size>0 else []
    return out

# 2) load each candidate and summarize
summaries = {}
for p in cand:
    print(f"\n=== Loading {p.name} ===")
    try:
        with open(p, "rb") as f:
            obj = pickle.load(f)
    except Exception as e:
        print("  ERROR loading:", e)
        continue
    summary = summarize_matrix(obj)
    summaries[p.name] = {"path": p, "obj": obj, "summary": summary}
    # print the summary nicely
    print("  shape:", summary["shape"])
    print("  dtype:", summary["dtype"])
    print("  min,max,mean:", summary["min"], summary["max"], summary["mean"])
    print("  p10,p25,p50,p75,p90:", summary["p10"], summary["p25"], summary["p50"], summary["p75"], summary["p90"])
    print("  binary_like_0_1:", summary["binary_like_0_1"])
    print("  all_values_integer:", summary["all_values_integer"])
    print("  unique_count (capped):", summary["unique_count"])
    if not summary["binary_like_0_1"]:
        print("  unique sample (first 10):", summary["unique_sample_first10"])

# 3) If more than one occ_skills matrix exists, compare a sample row/column to highlight differences
if len(summaries) > 1:
    keys = list(summaries.keys())
    print("\nMultiple occ_skills matrices found. Showing a quick comparison between the first two:")
    name_a, name_b = keys[0], keys[1]
    A = summaries[name_a]["obj"]
    B = summaries[name_b]["obj"]
    print(" -", name_a, "summary:", summaries[name_a]["summary"])
    print(" -", name_b, "summary:", summaries[name_b]["summary"])
    # choose a sample occupation row (first index)
    def get_row(obj, idx=0):
        if isinstance(obj, pd.DataFrame):
            return obj.iloc[idx].values
        else:
            return np.asarray(obj)[idx]
    try:
        row_a = get_row(A, 0)
        row_b = get_row(B, 0)
        # compute differences for that row
        diff = row_b - row_a
        print("\nSample row (occupation #0) comparison:")
        print(" rowA nonzero count:", int((row_a!=0).sum()), " rowB nonzero count:", int((row_b!=0).sum()))
        # show examples of positions where values differ
        diff_idx = np.where(np.abs(diff) > 1e-8)[0][:20]
        if diff_idx.size:
            print(" Example differing indices and values (first 20):")
            for i in diff_idx:
                print(f"  idx {i}: A={row_a[i]}  B={row_b[i]}  diff={diff[i]}")
        else:
            print(" No differences found in sample row.")
    except Exception as e:
        print("Could not compare rows:", e)

# 4) search the repository to find where occ_skills_matrix is referenced (which file name is loaded)
#    search under project root (assumes Trials notebook set project_root earlier). If not defined, fallback to current working dir.
try:
    project_root = Path().resolve().parents[0]
except Exception:
    project_root = Path.cwd()
print("\nSearching python files under project root for references to 'occ_skills_matrix' or 'occ_skills_matrix.pkl' ...")
matches = []
for p in project_root.rglob("*.py"):
    try:
        txt = p.read_text(encoding="utf8", errors="ignore")
        if "occ_skills_matrix" in txt or "occ_skills_matrix.pkl" in txt:
            # show a small snippet with line numbers
            for i, line in enumerate(txt.splitlines(), start=1):
                if "occ_skills_matrix" in line:
                    snippet = line.strip()
                    matches.append((p.relative_to(project_root), i, snippet))
    except Exception:
        continue
if matches:
    print("Found references in these files (file, line, snippet):")
    for m in matches[:50]:
        print(" -", m[0], "line", m[1], ":", m[2])
else:
    print("No matches found. You may be loading the matrix dynamically in a .ipynb or via a different key; try searching in notebooks too.")

# 5) quick check for CONFIG usage: search for WEIGHT_OPTIONAL_SKILL / WEIGHT_ESSENTIAL_SKILL in YAML or py files
print("\nSearching for weight config keys in repo (WEIGHT_OPTIONAL_SKILL / WEIGHT_ESSENTIAL_SKILL / WEIGHT_UNIFORM):")
cfg_matches = []
for p in project_root.rglob("*"):
    if p.suffix in {".yml", ".yaml", ".py", ".txt", ".cfg", ".json"}:
        try:
            txt = p.read_text(encoding="utf8", errors="ignore")
            if "WEIGHT_OPTIONAL_SKILL" in txt or "WEIGHT_ESSENTIAL_SKILL" in txt or "WEIGHT_UNIFORM" in txt:
                cfg_matches.append(p.relative_to(project_root))
        except Exception:
            pass
if cfg_matches:
    for c in sorted(set(cfg_matches)):
        print(" -", c)
else:
    print("No config references found in the repo search (maybe config is loaded from elsewhere).")

# 6) Short human-readable instructions for interpreting the summaries:
print(textwrap.dedent("""
\nINTERPRETING THE OUTPUTS
- If summary['binary_like_0_1'] is True and 'all_values_integer' is True for the loaded matrix,
  that matrix is essentially binary (0/1). That means the occ–skill matrix used is the essential-only (or thresholded) binary version.
- If summary shows non-integer values (e.g. min=0.0, max=1.0 but many values = 0.5 or other floats), then that matrix contains weights (optional skills are being counted with fractional weights).
- If you find multiple occ_skills_matrix files (e.g. occ_skills_matrix.pkl and occ_skills_matrix_weighted.pkl), check which filename appears in the simulation code (see the file matches printed above). The simulation uses whichever filename it opens.
- If the simulation opens occ_skills_matrix.pkl and that file is binary (0/1), then optional weights (e.g. 0.5) were NOT used in that run.
- If you want me to interpret the actual printed summaries, copy them here and I will read them and say exactly which matrix your notebook is using.
"""))
# ---------- end of cell ----------


Looking in: /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/processed/esco

Candidate occ_skills files found:
 - occ_skills_matrix.pkl

=== Loading occ_skills_matrix.pkl ===


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/1947996297.py:67: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  obj = pickle.load(f)


  shape: (3008, 13891)
  dtype: int64
  min,max,mean: 0.0 2.0 0.004370271888885655
  p10,p25,p50,p75,p90: 0.0 0.0 0.0 0.0 0.0
  binary_like_0_1: False
  all_values_integer: True
  unique_count (capped): 3
  unique sample (first 10): [0, 1, 2]

Searching python files under project root for references to 'occ_skills_matrix' or 'occ_skills_matrix.pkl' ...
No matches found. You may be loading the matrix dynamically in a .ipynb or via a different key; try searching in notebooks too.

Searching for weight config keys in repo (WEIGHT_OPTIONAL_SKILL / WEIGHT_ESSENTIAL_SKILL / WEIGHT_UNIFORM):
No config references found in the repo search (maybe config is loaded from elsewhere).


INTERPRETING THE OUTPUTS
- If summary['binary_like_0_1'] is True and 'all_values_integer' is True for the loaded matrix,
  that matrix is essentially binary (0/1). That means the occ–skill matrix used is the essential-only (or thresholded) binary version.
- If summary shows non-integer values (e.g. min=0.0, max=1.0 bu

In [5]:
# Paste & run
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

base_esco = Path(useful_paths.data_processed) / "esco"
p = base_esco / "occ_skills_matrix.pkl"
print("Loading:", p)
with open(p, "rb") as f:
    occ_skills = pickle.load(f)   # DataFrame rows=occ URIs, cols=skill URIs

# basic unique values
vals = np.unique(occ_skills.values.ravel())
print("Unique values in occ_skills_matrix.pkl:", vals)

# heuristic mapping (attempt to infer scale)
# If we see {0,1,2} and know WEIGHT_OPTIONAL_SKILL=0.5 / WEIGHT_ESSENTIAL_SKILL=1.0,
# then assume scale = 2 (so 2->essential, 1->optional).
assumed_scale = None
if set(vals) <= {0,1}:
    print("Matrix currently binary-like (0/1). Likely essential-only.")
elif set(vals) <= {0,1,2}:
    # try to infer scale by checking ratio of 2 to 1
    assumed_scale = 2
    print("Matrix contains 0/1/2 — likely weighted with scale=2 (essential=2, optional=1).")
else:
    print("Matrix contains other values; printing sample distribution.")

# counts of cells
total_cells = occ_skills.size
count_essential = int((occ_skills.values == 2).sum())
count_optional  = int((occ_skills.values == 1).sum())
count_none      = int((occ_skills.values == 0).sum())
print(f"Total cells: {total_cells:,}")
print(f"Essential cells (value==2): {count_essential:,}")
print(f"Optional cells  (value==1): {count_optional:,}")
print(f"Empty cells     (value==0): {count_none:,}")

# fraction of non-zero cells that are optional vs essential
nonzero_cells = count_essential + count_optional
if nonzero_cells:
    print("Of non-zero cells: optional fraction = "
          f"{count_optional/nonzero_cells:.3f}, essential fraction = {count_essential/nonzero_cells:.3f}")
else:
    print("No non-zero cells detected!")

# per-skill summary: how many occupations mark it essential vs optional
skill_idx = occ_skills.columns
skill_counts = pd.DataFrame({
    "skill_uri": skill_idx,
    "n_essential": (occ_skills == 2).sum(axis=0).astype(int).values,
    "n_optional": (occ_skills == 1).sum(axis=0).astype(int).values
})
# show top skills by essential count and by optional count
print("\nTop 10 skills by n_essential (skill_uri, essential_count):")
print(skill_counts.sort_values("n_essential", ascending=False).head(10)[["skill_uri","n_essential"]].to_string(index=False))
print("\nTop 10 skills by n_optional (skill_uri, optional_count):")
print(skill_counts.sort_values("n_optional", ascending=False).head(10)[["skill_uri","n_optional"]].to_string(index=False))

# per-occupation summary: how many essential/optional each occupation requires (helpful for B)
occ_idx = occ_skills.index
occ_counts = pd.DataFrame({
    "occ_uri": occ_idx,
    "n_essential": (occ_skills == 2).sum(axis=1).astype(int).values,
    "n_optional": (occ_skills == 1).sum(axis=1).astype(int).values
})
print("\nTop 10 occupations by essential green/digital-skill count (occ_uri, n_essential, n_optional):")
print(occ_counts.sort_values("n_essential", ascending=False).head(10).to_string(index=False))

# Helpers: derive clean float-weighted or essential-only matrices if you want to re-run sims:
# float-weighted (recover original configured weights, dividing by assumed_scale)
if assumed_scale == 2:
    occ_skills_weighted = occ_skills.astype(float) / 2.0   # yields values {0.0,0.5,1.0}
    occ_skills_essential_only = (occ_skills == 2).astype(int)
    print("\nCreated occ_skills_weighted (float) and occ_skills_essential_only (binary) in memory.")
    # optionally save to files:
    # with open(base_esco / "occ_skills_matrix_weighted_float.pkl", "wb") as f:
    #     pickle.dump(occ_skills_weighted, f)
    # with open(base_esco / "occ_skills_matrix_essential_only.pkl", "wb") as f:
    #     pickle.dump(occ_skills_essential_only, f)
else:
    print("\nScale not inferred. Not creating derived matrices automatically.")

# Optional: Search notebooks for occurrences of the filename to identify what your simulation loads
proj_root = Path.cwd().resolve().parents[0]  # adjust if needed
print("\nSearching notebooks under project root for references to 'occ_skills_matrix' ...")
nb_matches = []
for nb in proj_root.rglob("*.ipynb"):
    try:
        txt = nb.read_text(encoding="utf8", errors="ignore")
        if "occ_skills_matrix" in txt:
            nb_matches.append(nb.relative_to(proj_root))
    except Exception:
        pass
print("Notebooks referencing this name (sample):", nb_matches[:20])


Loading: /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/processed/esco/occ_skills_matrix.pkl


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/1977508528.py:12: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)   # DataFrame rows=occ URIs, cols=skill URIs


Unique values in occ_skills_matrix.pkl: [0 1 2]
Matrix contains 0/1/2 — likely weighted with scale=2 (essential=2, optional=1).
Total cells: 41,784,128
Essential cells (value==2): 58,756
Optional cells  (value==1): 65,096
Empty cells     (value==0): 41,660,276
Of non-zero cells: optional fraction = 0.526, essential fraction = 0.474

Top 10 skills by n_essential (skill_uri, essential_count):
                                                            skill_uri  n_essential
http://data.europa.eu/esco/skill/ccfbdaad-b91d-4bfe-bd0e-d30b33587f19          260
http://data.europa.eu/esco/skill/e54ff029-1ce9-447d-a5b2-eb7283a23e6e          216
http://data.europa.eu/esco/skill/1a3660c2-011c-4e96-9b1d-529afc305428          154
http://data.europa.eu/esco/skill/415abd43-e8e5-4643-b5da-5f11307af57a          148
http://data.europa.eu/esco/skill/9df34bc3-25d4-4452-a896-4d19b94ef896          145
http://data.europa.eu/esco/skill/cd5efa8c-e44d-4cbc-91c6-796018dbed68          136
http://data.europa.eu/esc

In [6]:
# Paste this at the end of the notebook you already use (you have useful_paths available)
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

base_esco = Path(useful_paths.data_processed) / "esco"
p = base_esco / "occ_skills_matrix.pkl"
print("Loading:", p)
with open(p, "rb") as f:
    occ_skills = pickle.load(f)   # DataFrame rows=occ URIs, cols=skill URIs

# infer mapping (same logic as before)
vals = np.unique(occ_skills.values.ravel())
print("Unique values:", vals)
scale = 2 if set(vals) <= {0,1,2} else 1
print("Assumed scale:", scale, "(so essential ->", scale, "optional ->", int(scale*0.5), ")")

# Build the three matrices we want to compare
occ_weighted_float = occ_skills.astype(float) / float(scale)          # {0.0, 0.5, 1.0}
occ_essential_only  = (occ_skills == scale).astype(int)               # {0,1} essential only
occ_binary_any      = (occ_skills > 0).astype(int)                    # any skill (optional or essential) as 1

# choose whether to compute full similarity matrix or a sampled check
n_occs = occ_skills.shape[0]
print("Occupations:", n_occs, "Skills:", occ_skills.shape[1])

def sim_stats(A):
    """Compute light-weight n x n similarity (dot-product of row vectors) summaries.
    Returns: dictionary with mean, median, fraction of zero similarities, mean max-per-occ.
    Warning: full n x n matrix for ~3k occs is ~9M entries (OK), but be mindful of memory/time."""
    M = A.values
    S = M @ M.T
    # zero-sim fraction (excluding diagonal)
    diag = np.diag(S).copy()
    S_no_diag = S.copy()
    np.fill_diagonal(S_no_diag, 0)
    n_pairs = n_occs * (n_occs - 1)
    frac_zero = (S_no_diag == 0).sum() / n_pairs
    return {
        "mean_sim": float(S_no_diag.mean()),
        "median_sim": float(np.median(S_no_diag)),
        "frac_zero_sim": float(frac_zero),
        "mean_max_sim_per_occ": float(S_no_diag.max(axis=1).mean())
    }

print("\nComputing similarity summaries (this computes the full n x n dot-product matrices)...")
stats_weighted = sim_stats(occ_weighted_float)
stats_essential = sim_stats(occ_essential_only)
stats_any = sim_stats(occ_binary_any)

pd.DataFrame([stats_weighted, stats_essential, stats_any], index=["weighted(0/0.5/1.0)","essential_only(0/1)","any_binary(0/1)"])

# Compare per-occupation max similarity differences (weighted vs essential-only)
M_w = occ_weighted_float.values
M_e = occ_essential_only.values
S_w = M_w @ M_w.T
S_e = M_e @ M_e.T
np.fill_diagonal(S_w, 0); np.fill_diagonal(S_e, 0)

max_w = S_w.max(axis=1)
max_e = S_e.max(axis=1)
delta_max = max_w - max_e

print("\nPer-occupation max similarity (weighted - essential):")
print(" mean delta:", delta_max.mean(), " median delta:", np.median(delta_max))
print(" fraction of occupations with increased max similarity under weighted matrix:", (delta_max>0).mean())

# Which occupations gained most from optional skills (top 10)
occ_index = occ_skills.index
top_gain_idx = np.argsort(-delta_max)[:10]
print("\nTop 10 occupations that increase their max-sim when optional skills included:")
for i in top_gain_idx:
    print(f" - {occ_index[i]}  delta_max={delta_max[i]:.2f}  (max_e={max_e[i]:.2f} -> max_w={max_w[i]:.2f})")

# Optional: find how many additional non-zero similarity pairs appear when including optional skills
pairs_e_nonzero = (S_e > 0).sum()
pairs_w_nonzero = (S_w > 0).sum()
print("\nNon-zero similarity pairs: essential-only:", pairs_e_nonzero, " weighted:", pairs_w_nonzero,
      "  increase fraction:", (pairs_w_nonzero - pairs_e_nonzero) / max(1, pairs_e_nonzero))

# If you want a *thresholded* test (e.g. does optional inclusion push pairs over a threshold),
# set your threshold t and compute counts.
t = 1.0   # example threshold: adjust to the transition threshold you use in the simulation
pairs_over_t_e = (S_e >= t).sum()
pairs_over_t_w = (S_w >= t).sum()
print(f"\nPairs with similarity >= {t}: essential-only={pairs_over_t_e}, weighted={pairs_over_t_w}, delta={pairs_over_t_w - pairs_over_t_e}")

# Keep the matrices in memory if you want to run the same simulation logic after swapping them in:
# occ_skills_weighted_float  -> use for "optional included"
# occ_skills_essential_only  -> use for "essential-only"


Loading: /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/processed/esco/occ_skills_matrix.pkl


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/964099772.py:12: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)   # DataFrame rows=occ URIs, cols=skill URIs


Unique values: [0 1 2]
Assumed scale: 2 (so essential -> 2 optional -> 1 )
Occupations: 3008 Skills: 13891

Computing similarity summaries (this computes the full n x n dot-product matrices)...

Per-occupation max similarity (weighted - essential):
 mean delta: 5.460688164893617  median delta: 4.75
 fraction of occupations with increased max similarity under weighted matrix: 0.9936835106382979

Top 10 occupations that increase their max-sim when optional skills included:
 - http://data.europa.eu/esco/occupation/3af4d6be-90b5-41f6-be06-3330dab3df73  delta_max=28.25  (max_e=2.00 -> max_w=30.25)
 - http://data.europa.eu/esco/occupation/3d616092-b4fd-4c14-b0e9-a9119bbbd171  delta_max=26.25  (max_e=13.00 -> max_w=39.25)
 - http://data.europa.eu/esco/occupation/19f6634f-ec0c-4513-b0d9-d43f9ddeb82d  delta_max=24.75  (max_e=17.00 -> max_w=41.75)
 - http://data.europa.eu/esco/occupation/4b05bef2-ded4-4b09-ab94-f4c6a555a775  delta_max=24.25  (max_e=36.00 -> max_w=60.25)
 - http://data.europa.eu/

In [8]:
# ---- Paste this in your notebook and run ----
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

# assume occ_skills, occ_ess_only, occ_weighted, M_e_full were already created by previous cell
# if any missing, try to (re)create them:
try:
    occ_skills
except NameError:
    p = Path(useful_paths.data_processed) / "esco" / "occ_skills_matrix.pkl"
    with open(p, "rb") as f:
        occ_skills = pickle.load(f)
    occ_ess_only = (occ_skills == 2).astype(int)
    occ_weighted  = occ_skills.astype(float) / 2.0

# ensure the canonical matrices exist
if 'occ_ess_only' not in globals():
    occ_ess_only = (occ_skills == 2).astype(int)
if 'occ_weighted' not in globals():
    occ_weighted = occ_skills.astype(float) / 2.0

# 1) recreate digital_uris / green_uris if missing (safe, idempotent)
try:
    digital_uris, green_uris
except NameError:
    # try to use already-loaded df_dig/df_grn
    if 'df_dig' in globals() and 'df_grn' in globals():
        digital_uris = df_dig["conceptUri"].tolist()
        green_uris   = df_grn["conceptUri"].tolist()
    else:
        # load from raw files (adjust path only if yours differ)
        p_d = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "digCompSkillsCollection_en.csv"
        p_g = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "greenSkillsCollection_en.csv"
        df_dig = pd.read_csv(p_d, index_col=0)
        df_grn = pd.read_csv(p_g, index_col=0)
        digital_uris = df_dig["conceptUri"].tolist()
        green_uris   = df_grn["conceptUri"].tolist()

print("Found digital_uris:", len(digital_uris), "Found green_uris:", len(green_uris))

# --- helper: sim matrix builder for a given df slice (rows=occ) ---
def sim_matrix(df):
    A = df.values
    M = A @ A.T
    np.fill_diagonal(M, 0.0)
    return M

# full essential baseline if needed
M_e_full = sim_matrix(occ_ess_only)

# 2) find which digital/green columns are actually present in occ_skills
dig_cols = [u for u in digital_uris if u in occ_skills.columns]
grn_cols = [u for u in green_uris if u in occ_skills.columns]
print("#digital cols present in occ_skills:", len(dig_cols))
print("#green   cols present in occ_skills:", len(grn_cols))

# 3) build restricted matrices & summarize (digital-only and green-only)
M_e_dig = sim_matrix(occ_ess_only[dig_cols]) if len(dig_cols)>0 else np.zeros_like(M_e_full)
M_w_dig = sim_matrix(occ_weighted[dig_cols])  if len(dig_cols)>0 else np.zeros_like(M_e_full)
M_e_grn = sim_matrix(occ_ess_only[grn_cols]) if len(grn_cols)>0 else np.zeros_like(M_e_full)
M_w_grn = sim_matrix(occ_weighted[grn_cols])  if len(grn_cols)>0 else np.zeros_like(M_e_full)

def summarize_pair(M_base, M_group, label):
    max_base = M_base.max(axis=1)
    max_group = M_group.max(axis=1)
    delta = max_group - max_base
    print(f"\nSummary for {label}:")
    print("  mean(delta max) = ", float(delta.mean()))
    print("  fraction occupations with increase >0:", float((delta > 1e-8).mean()))
    print("  90th percentile of delta:", float(np.percentile(delta, 90)))

summarize_pair(M_e_full, M_e_dig, "digital (essential cols only vs full essential baseline)")
summarize_pair(M_e_full, M_w_dig, "digital (weighted cols vs full essential baseline)")
summarize_pair(M_e_full, M_e_grn, "green (essential cols only vs full essential baseline)")
summarize_pair(M_e_full, M_w_grn, "green (weighted cols vs full essential baseline)")

# 4) cumulative-add test for a single test occupation
test_occ = occ_skills.index[0]   # change to any occupation URI you want to test
print("\nTest occupation for cumulative-add:", test_occ)

# source baseline as essential/binary
v_base = occ_ess_only.loc[test_occ].values.astype(float)

order = dig_cols.copy()  # order of digital skills to add

def cumulative_add_sim(order, treat_as="optional", target='essential'):
    if target == 'essential':
        targets = occ_ess_only.values
    else:
        targets = occ_weighted.values
    v = v_base.copy()
    sims = []
    sims.append(float((targets @ v).max()))
    for skill in order:
        add_val = 0.5 if treat_as == "optional" else 1.0
        # set cumulatively
        col_idx = list(occ_skills.columns).index(skill)
        v[col_idx] = max(v[col_idx], add_val)
        sims.append(float((targets @ v).max()))
    return sims

sims_opt = cumulative_add_sim(order, treat_as="optional")
sims_ess = cumulative_add_sim(order, treat_as="essential")

print("\nCumulative-add progression (optional): first 10 steps:", sims_opt[:11])
print("Monotonic (optional):", all(x<=y+1e-12 for x,y in zip(sims_opt, sims_opt[1:])))
print("\nCumulative-add progression (essential): first 10 steps:", sims_ess[:11])
print("Monotonic (essential):", all(x<=y+1e-12 for x,y in zip(sims_ess, sims_ess[1:])))

# 5) threshold crossing check (use your q_viable if available)
q_viable = 3.68
def steps_to_cross(sims, threshold=q_viable):
    for i, val in enumerate(sims):
        if val >= threshold:
            return i
    return None

print("\nSteps to cross q_viable (optional):", steps_to_cross(sims_opt))
print("Steps to cross q_viable (essential):", steps_to_cross(sims_ess))

# 6) final helpful note
print("\nIf the digital-weighted summary shows very small mean(delta) and small fraction increased,"
      " then the digital columns are not doing much. If the weighted-green summary shows much larger"
      " effects, green columns are providing the connectivity. The cumulative-add (optional vs essential)"
      " shows whether making digital skills essential would help cross thresholds.")


Found digital_uris: 21 Found green_uris: 570
#digital cols present in occ_skills: 21
#green   cols present in occ_skills: 570

Summary for digital (essential cols only vs full essential baseline):
  mean(delta max) =  -11.269946808510639
  fraction occupations with increase >0: 0.0
  90th percentile of delta: -1.0

Summary for digital (weighted cols vs full essential baseline):
  mean(delta max) =  -11.265126329787234
  fraction occupations with increase >0: 0.0003324468085106383
  90th percentile of delta: -1.0

Summary for green (essential cols only vs full essential baseline):
  mean(delta max) =  -10.543218085106384
  fraction occupations with increase >0: 0.0
  90th percentile of delta: -1.0

Summary for green (weighted cols vs full essential baseline):
  mean(delta max) =  -10.276928191489361
  fraction occupations with increase >0: 0.02925531914893617
  90th percentile of delta: -0.5

Test occupation for cumulative-add: http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87

In [11]:
# ---- paste this cell ----
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import warnings

# reuse useful_paths from your notebook
base_processed = Path(useful_paths.data_processed) / "esco"
base_raw = Path(useful_paths.data_raw) / "esco" / "v1.1.0"

p_occ_skills = base_processed / "occ_skills_matrix.pkl"
p_dig = base_raw / "digCompSkillsCollection_en.csv"
p_grn = base_raw / "greenSkillsCollection_en.csv"
p_occs = base_raw / "occupationsCollection_en.csv"

# 1) load occ_skills
with open(p_occ_skills, "rb") as f:
    occ_skills = pickle.load(f)
assert isinstance(occ_skills, pd.DataFrame)

# 2) load digital & green lists (these should exist)
df_dig = pd.read_csv(p_dig, low_memory=False)
df_grn = pd.read_csv(p_grn, low_memory=False)

# 3) obtain occupation metadata: try CSV first, else fall back to Esco() helper
try:
    df_occs = pd.read_csv(p_occs, low_memory=False)
    print(f"Loaded occupations CSV from {p_occs}")
except FileNotFoundError:
    try:
        from data.framework import Esco
        esco = Esco()
        # esco.occupations should be a DataFrame-like
        df_occs = esco.occupations.reset_index()
        print("Loaded occupations from Esco() object (fallback).")
    except Exception as e:
        df_occs = None
        warnings.warn("Occupations metadata not found in CSV and Esco() fallback failed; ISCO-aggregations will be skipped.")

# helper: detect label and isco columns robustly
def pick_label_col(df):
    for c in ("preferredLabel","preferred_label","label","preferred.label","preferredlabel"):
        if c in df.columns:
            return c
    return None

# 4) basic diagnostics of occ_skills
vals = np.unique(occ_skills.values)
print("\n=== occ_skills diagnostics ===")
print("shape:", occ_skills.shape)
print("dtype:", occ_skills.values.dtype)
print("unique values (sample):", vals.tolist())

# treat value==2 as essential, 1 as optional if present
ess_mask = (occ_skills == 2)
opt_mask = (occ_skills == 1)

# 5) digital/green columns present
digital_uris = df_dig["conceptUri"].tolist()
green_uris = df_grn["conceptUri"].tolist()
dig_cols = [u for u in digital_uris if u in occ_skills.columns]
grn_cols = [u for u in green_uris if u in occ_skills.columns]
print(f"\nFound digital_uris: {len(digital_uris)} Found green_uris: {len(green_uris)}")
print(f"#digital cols present in occ_skills: {len(dig_cols)}")
print(f"#green   cols present in occ_skills: {len(grn_cols)}")

# 6) prepare occupation metadata if available
occ_info = None
if df_occs is not None:
    # standardise: ensure conceptUri exists and use it as index
    if "conceptUri" not in df_occs.columns and df_occs.index.name == "conceptUri":
        df_occs = df_occs.reset_index()
    if "conceptUri" in df_occs.columns:
        occ_info = df_occs.set_index("conceptUri", drop=False)
    else:
        warnings.warn("occupations metadata has no 'conceptUri' column; ISCO aggregations will be skipped.")
        occ_info = None

# detect ISCO column name and create isco4 / isco3 if possible
if occ_info is not None:
    if "iscoGroup" in occ_info.columns:
        isco_col = "iscoGroup"
    elif "code" in occ_info.columns:
        isco_col = "code"
    else:
        isco_col = None
    if isco_col:
        # coerce to int-ish where possible; handle missing gracefully
        def parse_isco(x):
            try:
                s = str(x).strip()
                # if already like '3122' or '312' keep, else extract digits
                import re
                m = re.search(r"(\d{3,4})$", s)
                if m:
                    return int(m.group(1))
                # fallback
                return int(float(s))
            except Exception:
                return np.nan
        occ_info["isco4"] = occ_info[isco_col].astype(str).apply(parse_isco)
        occ_info["isco3"] = occ_info["isco4"].dropna().astype(int) // 10
    else:
        warnings.warn("Could not find an ISCO code column in occupations metadata; ISCO grouping skipped.")
        occ_info["isco4"] = np.nan
        occ_info["isco3"] = np.nan
    occ_label_col = pick_label_col(occ_info)
else:
    occ_label_col = None

# 7) per-skill ISCO coverage helper (counts number of distinct ISCO-4 occs and distinct ISCO-3 groups where skill is essential)
def per_skill_isco_counts(skill_cols):
    rows = []
    for s in skill_cols:
        if s not in occ_skills.columns:
            rows.append((s, 0, 0))
            continue
        # occupations where skill s is essential (value==2)
        occs_with = occ_skills.index[ess_mask[s] == True] if s in occ_skills.columns else pd.Index([])
        n_isco4 = 0
        n_isco3 = 0
        if occ_info is not None:
            occs_with = [o for o in occs_with if o in occ_info.index]
            n_isco4 = len(occs_with)
            if n_isco4 > 0:
                n_isco3 = occ_info.loc[occs_with, "isco3"].dropna().astype(int).nunique()
        rows.append((s, n_isco4, n_isco3))
    return pd.DataFrame(rows, columns=["skill_uri","n_isco4","n_isco3"]).set_index("skill_uri")

dig_df = per_skill_isco_counts(dig_cols)
grn_df = per_skill_isco_counts(grn_cols)

def summarize(df):
    return {
        "n_skills": int(len(df)),
        "ever_essential": int((df["n_isco4"] > 0).sum()),
        "median_isco4": int(df["n_isco4"].median()) if len(df)>0 else 0,
        "max4": int(df["n_isco4"].max()) if len(df)>0 else 0,
        "max3": int(df["n_isco3"].max()) if len(df)>0 else 0
    }

dig_s = summarize(dig_df)
grn_s = summarize(grn_df)

print("\nSummary stats (per-skill coverage at ISCO-4 / ISCO-3):")
print("Digital:", dig_s)
print("Green:  ", grn_s)

# 8) top skills by ISCO-4 essential coverage (with labels if available in the original lists)
def top_skills(df, topn=5, label_df=None):
    out = []
    for uri, row in df.sort_values("n_isco4", ascending=False).head(topn).iterrows():
        label = None
        if label_df is not None and "conceptUri" in label_df.columns:
            hit = label_df[label_df["conceptUri"] == uri]
            if not hit.empty:
                possible_label_col = pick_label_col(label_df)
                if possible_label_col is not None:
                    label = hit.iloc[0][possible_label_col]
        out.append((uri, label, int(row["n_isco4"]), int(row["n_isco3"])))
    return out

dig_label_df = df_dig if "conceptUri" in df_dig.columns else None
grn_label_df = df_grn if "conceptUri" in df_grn.columns else None

print("\nTop digital skills by number of ISCO-4 occupations where they are ESSENTIAL:")
for uri, label, n4, n3 in top_skills(dig_df, topn=10, label_df=dig_label_df):
    print(f" • {label or uri}  ({uri}) — essential in {n4} ISCO-4 occs / {n3} ISCO-3 groups")

print("\nTop green skills by number of ISCO-4 occupations where they are ESSENTIAL:")
for uri, label, n4, n3 in top_skills(grn_df, topn=10, label_df=grn_label_df):
    print(f" • {label or uri}  ({uri}) — essential in {n4} ISCO-4 occs / {n3} ISCO-3 groups")

# 9) top occupations (ISCO-4) by essential counts for digital & green (if occ_info available)
if occ_info is not None:
    occ_common = occ_skills.index.intersection(occ_info.index)
    occ_green_counts = occ_skills.loc[occ_common, grn_cols].eq(2).sum(axis=1).sort_values(ascending=False)
    occ_dig_counts = occ_skills.loc[occ_common, dig_cols].eq(2).sum(axis=1).sort_values(ascending=False)

    print("\nTop 10 occupations (ISCO-4) by number of essential GREEN skills:")
    for uri, n in occ_green_counts.head(10).items():
        lbl = occ_info.loc[uri, occ_label_col] if occ_label_col in occ_info.columns else uri
        print(f" • {lbl} ({uri}) — {int(n)} green skills")

    print("\nTop 10 occupations (ISCO-4) by number of essential DIGITAL skills:")
    for uri, n in occ_dig_counts.head(10).items():
        lbl = occ_info.loc[uri, occ_label_col] if occ_label_col in occ_info.columns else uri
        print(f" • {lbl} ({uri}) — {int(n)} digital skills")

    # ISCO-3 aggregation
    occ_info_for_counts = occ_info.loc[occ_common].copy()
    occ_info_for_counts["green_ess"] = occ_green_counts
    occ_info_for_counts["dig_ess"] = occ_dig_counts

    isco3_green_sum = occ_info_for_counts.groupby("isco3")["green_ess"].sum().dropna().sort_values(ascending=False)
    isco3_dig_sum = occ_info_for_counts.groupby("isco3")["dig_ess"].sum().dropna().sort_values(ascending=False)

    print("\nTop 10 ISCO-3 groups by TOTAL essential GREEN skills across occupations (sum of ess counts):")
    for isco3, tot in isco3_green_sum.head(10).items():
        print(f" • ISCO-3 {int(isco3)} — total essential-green-skills = {int(tot)}; occupations with any essential-green = {(occ_info_for_counts.groupby('isco3')['green_ess'].apply(lambda s:(s>0).sum()).get(isco3,0))}")

    print("\nTop 10 ISCO-3 groups by TOTAL essential DIGITAL skills across occupations (sum of ess counts):")
    for isco3, tot in isco3_dig_sum.head(10).items():
        print(f" • ISCO-3 {int(isco3)} — total essential-dig-skills = {int(tot)}; occupations with any essential-digital = {(occ_info_for_counts.groupby('isco3')['dig_ess'].apply(lambda s:(s>0).sum()).get(isco3,0))}")
else:
    print("\nOccupation metadata not available: skipped per-occupation and ISCO group summaries.")

# compact summary for table use
print("\nCompact summary numbers you can paste into a table:")
print("Digital: #skills, #ever_essential, median_isco4, max4/max3 =",
      dig_s["n_skills"], dig_s["ever_essential"], dig_s["median_isco4"], f"{dig_s['max4']}/{dig_s['max3']}")
print("Green:   #skills, #ever_essential, median_isco4, max4/max3 =",
      grn_s["n_skills"], grn_s["ever_essential"], grn_s["median_isco4"], f"{grn_s['max4']}/{grn_s['max3']}")
# ---- end cell ----


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/1336567324.py:19: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)


Loaded occupations from Esco() object (fallback).

=== occ_skills diagnostics ===
shape: (3008, 13891)
dtype: int64
unique values (sample): [0, 1, 2]

Found digital_uris: 21 Found green_uris: 570
#digital cols present in occ_skills: 21
#green   cols present in occ_skills: 570

Summary stats (per-skill coverage at ISCO-4 / ISCO-3):
Digital: {'n_skills': 21, 'ever_essential': 9, 'median_isco4': 0, 'max4': 32, 'max3': 13}
Green:   {'n_skills': 570, 'ever_essential': 473, 'median_isco4': 3, 'max4': 68, 'max3': 23}

Top digital skills by number of ISCO-4 occupations where they are ESSENTIAL:
 • computer programming  (http://data.europa.eu/esco/skill/21d2f96d-35f7-4e3f-9745-c533d2dd6e97) — essential in 32 ISCO-4 occs / 11 ISCO-3 groups
 • solve technical problems  (http://data.europa.eu/esco/skill/14832d87-2f2f-4895-b290-e4760ebae42a) — essential in 25 ISCO-4 occs / 13 ISCO-3 groups
 • identify digital competence gaps  (http://data.europa.eu/esco/skill/238343b1-7b51-42b3-a9ed-cf24d3a236e7) —

In [12]:
# ---------- run this at the bottom of your notebook ----------
import os
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data.framework import Esco

out_dir = Path(useful_paths.output_dir) if hasattr(useful_paths, "output_dir") else Path("figures")
out_dir.mkdir(parents=True, exist_ok=True)

# 1) load occ_skills matrix (same file you used previously)
p_mat = Path(useful_paths.data_processed) / "esco" / "occ_skills_matrix.pkl"
print("Loading occ-skills matrix from:", p_mat)
with open(p_mat, "rb") as f:
    occ_skills = pickle.load(f)   # DataFrame rows=occ URIs, cols=skill URIs
print("occ_skills shape:", occ_skills.shape, "unique values sample:", np.unique(occ_skills.values)[:10])

# essential mask (ESCO encoding in your repo: essential == 2)
ess_mask = (occ_skills == 2)

# 2) load digital / green skill lists (try typical paths; fallback to variables if present)
p_dig = Path(useful_paths.data_raw) / "esco" / useful_paths.ESCO_VERSION if hasattr(useful_paths,"ESCO_VERSION") else Path("esco") / "v1.1.0"
# common filenames you used before:
p_dig = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "digCompSkillsCollection_en.csv"
p_grn = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "greenSkillsCollection_en.csv"

digital_uris = None
green_uris = None
if p_dig.exists() and p_grn.exists():
    df_dig = pd.read_csv(p_dig, index_col=0)
    df_grn = pd.read_csv(p_grn, index_col=0)
    # fall back to common column name "conceptUri"
    digital_uris = df_dig["conceptUri"].tolist()
    green_uris = df_grn["conceptUri"].tolist()
    print("Loaded digital/green lists from CSVs:", p_dig.name, p_grn.name)
else:
    # fallback: check if variables exist in this kernel
    if "digital_uris" in globals():
        digital_uris = globals()["digital_uris"]
    if "green_uris" in globals():
        green_uris = globals()["green_uris"]
    print("CSV digital/green lists not found on disk; using in-kernel variables if present.")

# final fallback: use Esco() object lists if available
esco = Esco()
if digital_uris is None:
    try:
        # Esco object sometimes stores collections; try plausible paths
        if hasattr(esco, "digital_skills"):
            digital_uris = esco.digital_skills["conceptUri"].tolist()
        else:
            # try reading from esco.skills and filtering by some tag - skip: leave None
            digital_uris = []
    except Exception:
        digital_uris = []
if green_uris is None:
    try:
        if hasattr(esco, "green_skills"):
            green_uris = esco.green_skills["conceptUri"].tolist()
        else:
            green_uris = []
    except Exception:
        green_uris = []

# Ensure digital_uris / green_uris intersect with occ_skills columns
dig_cols = [u for u in (digital_uris or []) if u in occ_skills.columns]
grn_cols = [u for u in (green_uris or []) if u in occ_skills.columns]
print(f"Found digital_uris: {len(digital_uris or [])} Found green_uris: {len(green_uris or [])}")
print(f"#digital cols present in occ_skills: {len(dig_cols)}\n#green   cols present in occ_skills: {len(grn_cols)}")

# 3) occupation metadata -> map each occupation row to ISCO-4 and ISCO-3 codes
# Try to get mapping from Esco() object (esco.occupations)
df_occ_meta = None
try:
    df_occ_meta = esco.occupations.copy()
    # possible column names seen in your repo: 'conceptUri', 'iscoGroup', 'code', 'iscoGroupUri'
    if "iscoGroup" in df_occ_meta.columns:
        isco_col = "iscoGroup"
    elif "code" in df_occ_meta.columns:
        isco_col = "code"
    elif "iscoGroupUri" in df_occ_meta.columns:
        isco_col = "iscoGroupUri"
    else:
        isco_col = None
    if isco_col is None:
        raise KeyError("No isco column in esco.occupations")
    # normalize to 4-digit string code (remove non-digits and take last 4 digits if needed)
    def norm_isco4(v):
        s = str(v)
        digits = "".join(ch for ch in s if ch.isdigit())
        if len(digits) >= 4:
            return digits[-4:]  # often the last 4 are the ISCO-4
        return digits.zfill(4)
    df_occ_meta = df_occ_meta.set_index("conceptUri")
    df_occ_meta["isco4"] = df_occ_meta[isco_col].astype(str).apply(norm_isco4)
    df_occ_meta["isco3"] = df_occ_meta["isco4"].str[:3]
    # build mapping for occ_skills.index
    occ_to_isco4 = occ_skills.index.to_series().map(df_occ_meta["isco4"]).fillna("0000")
    occ_to_isco3 = occ_skills.index.to_series().map(df_occ_meta["isco3"]).fillna("000")
    print("Mapped occupations to ISCO-4/ISCO-3 (via esco.occupations).")
except Exception as e:
    print("Could not map via esco.occupations:", e)
    # fallback: try to derive code from occ URI (last 4 chars if numeric)
    def try_uri_code(u):
        s = str(u).rstrip("/")
        tail = s.split("/")[-1]
        digits = "".join(ch for ch in tail if ch.isdigit())
        if len(digits) >= 3:
            return digits[-4:].zfill(4), digits[-4:].zfill(4)[:3]
        return "0000","000"
    occ_to_isco4 = occ_skills.index.to_series().apply(lambda u: try_uri_code(u)[0])
    occ_to_isco3 = occ_skills.index.to_series().apply(lambda u: try_uri_code(u)[1])
    print("Fallback mapping applied using occupation URI tails.")

# 4) per-skill: number of ISCO-4 occupations where skill is essential (count ISCO-4 units)
per_skill_isco4_count = ess_mask.sum(axis=0).reindex(occ_skills.columns).fillna(0).astype(int)
# per-skill: number of distinct ISCO-3 groups where skill is essential (count unique groups with at least one occ essential)
ess_bool = ess_mask.astype(bool)
# group rows by isco3; for each group and skill, check any() (True if any sub-occupation requires it essential)
grp_any = ess_bool.groupby(occ_to_isco3).any()            # DataFrame indexed by isco3, cols skills -> True/False
per_skill_isco3_count = grp_any.sum(axis=0).reindex(occ_skills.columns).fillna(0).astype(int)

# 5) compute compact summaries for digital and green at ISCO-4 and ISCO-3 levels
def make_summary(skill_cols):
    cols = [c for c in skill_cols if c in occ_skills.columns]
    n_skills = len(skill_cols)
    ever_essential_isco4 = (per_skill_isco4_count[cols] > 0).sum() if cols else 0
    median_isco4 = int(per_skill_isco4_count[cols].median()) if cols else 0
    max4 = int(per_skill_isco4_count[cols].max()) if cols else 0
    max3 = int(per_skill_isco3_count[cols].max()) if cols else 0
    return {"n_skills": n_skills, "ever_essential": int(ever_essential_isco4),
            "median_isco4": median_isco4, "max4": max4, "max3": max3,
            "per_skill_isco4_counts": per_skill_isco4_count[cols] if cols else pd.Series(dtype=int),
            "per_skill_isco3_counts": per_skill_isco3_count[cols] if cols else pd.Series(dtype=int)}

dig_summary = make_summary(digital_uris or [])
grn_summary = make_summary(green_uris or [])

print("\nCompact summary numbers (paste into table):")
print("Digital: #skills, #ever_essential, median_isco4, max4/max3 =",
      dig_summary["n_skills"], dig_summary["ever_essential"],
      dig_summary["median_isco4"], f"{dig_summary['max4']}/{dig_summary['max3']}")
print("Green:   #skills, #ever_essential, median_isco4, max4/max3 =",
      grn_summary["n_skills"], grn_summary["ever_essential"],
      grn_summary["median_isco4"], f"{grn_summary['max4']}/{grn_summary['max3']}")

# 6) Top ISCO-3 groups by total essential green/digital skills (sum across occupations)
# total essential-green-skills per ISCO-3 = sum over occ in group of number of essential green skills in that occ
if grn_cols:
    per_occ_grn_ess_count = ess_mask[grn_cols].sum(axis=1)
    grn_by_isco3_total = per_occ_grn_ess_count.groupby(occ_to_isco3).sum().sort_values(ascending=False)
    grn_by_isco3_n_occs = (per_occ_grn_ess_count.gt(0).groupby(occ_to_isco3).sum()).reindex(grn_by_isco3_total.index)
else:
    grn_by_isco3_total = pd.Series(dtype=int)
    grn_by_isco3_n_occs = pd.Series(dtype=int)

if dig_cols:
    per_occ_dig_ess_count = ess_mask[dig_cols].sum(axis=1)
    dig_by_isco3_total = per_occ_dig_ess_count.groupby(occ_to_isco3).sum().sort_values(ascending=False)
    dig_by_isco3_n_occs = (per_occ_dig_ess_count.gt(0).groupby(occ_to_isco3).sum()).reindex(dig_by_isco3_total.index)
else:
    dig_by_isco3_total = pd.Series(dtype=int)
    dig_by_isco3_n_occs = pd.Series(dtype=int)

print("\nTop 10 ISCO-3 groups by TOTAL essential GREEN skills (total / n occs-with-any):")
for idx, val in grn_by_isco3_total.head(10).items():
    print(f" ISCO-3 {idx} — total essential-green-skills = {int(val)}; occupations with any essential-green = {int(grn_by_isco3_n_occs[idx])}")

print("\nTop 10 ISCO-3 groups by TOTAL essential DIGITAL skills (total / n occs-with-any):")
for idx, val in dig_by_isco3_total.head(10).items():
    print(f" ISCO-3 {idx} — total essential-dig-skills = {int(val)}; occupations with any essential-dig = {int(dig_by_isco3_n_occs[idx])}")

# 7) Top ISCO-4 occupations by number of essential green/digital skills (for the subtable)
top10_occ_green = ess_mask[grn_cols].sum(axis=1).nlargest(10)
top10_occ_dig   = ess_mask[dig_cols].sum(axis=1).nlargest(10)
# try to get preferredLabels for those occs (via esco.occupations)
occ_label_map = {}
try:
    occ_label_map = esco.occupations.set_index("conceptUri")["preferredLabel"].to_dict()
except Exception:
    pass

def pretty_occ_list(series):
    rows = []
    for occ_uri, n in series.items():
        label = occ_label_map.get(occ_uri, occ_uri)
        rows.append((label, occ_uri, int(n)))
    return rows

top10_green_pretty = pretty_occ_list(top10_occ_green)
top10_dig_pretty = pretty_occ_list(top10_occ_dig)

print("\nTop 10 ISCO-4 occupations by #essential GREEN skills (sample):")
for lab, uri, n in top10_green_pretty[:10]:
    print(" •", lab, "(", uri, ") —", n, "green skills")

print("\nTop 10 ISCO-4 occupations by #essential DIGITAL skills (sample):")
for lab, uri, n in top10_dig_pretty[:10]:
    print(" •", lab, "(", uri, ") —", n, "digital skills")

# 8) produce two plots:
# A) histogram of per-skill ISCO-3 essential counts (digital vs green)
plt.figure(figsize=(6,4))
if len(dig_summary["per_skill_isco3_counts"])>0:
    plt.hist(dig_summary["per_skill_isco3_counts"].values, bins=range(0, max(5, dig_summary["per_skill_isco3_counts"].max()+2)),
             alpha=0.6, label="Digital (per-skill # ISCO-3 groups)")
if len(grn_summary["per_skill_isco3_counts"])>0:
    plt.hist(grn_summary["per_skill_isco3_counts"].values, bins=30, alpha=0.6, label="Green (per-skill # ISCO-3 groups)")
plt.xlabel("Number of ISCO-3 groups where skill is ESSENTIAL")
plt.ylabel("Count of skills")
plt.legend()
plt.title("Distribution of per-skill essential coverage (ISCO-3)")
fn = out_dir / "per_skill_essential_isco3_hist.png"
plt.tight_layout()
plt.savefig(fn, dpi=150)
plt.close()
print("Saved:", fn)

# B) bar chart: top 10 ISCO-3 groups by essential-green total and essential-digital total (side-by-side)
topN = 10
top_grn = grn_by_isco3_total.head(topN)
top_dig = dig_by_isco3_total.head(topN)
# union index
ix = sorted(set(top_grn.index.tolist()) | set(top_dig.index.tolist()))
df_plot = pd.DataFrame({
    "green_total": grn_by_isco3_total.reindex(ix).fillna(0).astype(int),
    "digital_total": dig_by_isco3_total.reindex(ix).fillna(0).astype(int)
})
df_plot = df_plot.sort_values("green_total", ascending=False).head(topN)
plt.figure(figsize=(8,4.5))
ind = np.arange(len(df_plot))
width = 0.38
plt.bar(ind - width/2, df_plot["green_total"], width, label="Green (total essential)")
plt.bar(ind + width/2, df_plot["digital_total"], width, label="Digital (total essential)")
plt.xticks(ind, df_plot.index, rotation=45, ha="right")
plt.ylabel("Total essential-skill occurrences (sum across occupations)")
plt.title("Top ISCO-3 groups: essential GREEN vs DIGITAL skill totals")
plt.legend()
plt.tight_layout()
fn2 = out_dir / "top_isco3_essential_green_vs_dig.png"
plt.savefig(fn2, dpi=150)
plt.close()
print("Saved:", fn2)

# 9) produce LaTeX table string (filled with computed numbers)
latex_table = rf"""
\begin{{table}}[ht]
\centering
\caption{{Essential‐skill coverage and top occupations for digital vs.\ green skills (ISCO-3 aggregates shown)}}
\label{{tab:essential_skill_coverage_and_top_occ_isco3}}
\begin{{subtable}}[t]{{.48\textwidth}}
  \centering
  \caption{{Skill coverage across occupations}}
  \begin{{tabular}}{{lccc}}
    \toprule
                  & \# skills & \# ever essential & max \# occs (4-digit / 3-digit) \\
    \midrule
    Digital       & {dig_summary['n_skills']:>6} & {dig_summary['ever_essential']:>16} & {dig_summary['max4']}/{dig_summary['max3']} \\
    Green         & {grn_summary['n_skills']:>6} & {grn_summary['ever_essential']:>16} & {grn_summary['max4']}/{grn_summary['max3']} \\
    \bottomrule
  \end{{tabular}}
\end{{subtable}}%
\hfill
\begin{{subtable}}[t]{{.48\textwidth}}
  \centering
  \caption{{Top 5 ISCO-4 occupations by essential‐skill count (examples)}}
  \begin{{tabular}}{{lcc}}
    \toprule
    Occupation (example) & \# green & \# digital \\
    \midrule
"""
# take top 5 from earlier lists for the LaTeX subtable
for i in range(5):
    g_lab = top10_green_pretty[i][0] if i < len(top10_green_pretty) else ""
    g_n   = top10_green_pretty[i][2] if i < len(top10_green_pretty) else 0
    d_lab = top10_dig_pretty[i][0] if i < len(top10_dig_pretty) else ""
    d_n   = top10_dig_pretty[i][2] if i < len(top10_dig_pretty) else 0
    latex_table += f"    {g_lab} & {g_n} & {d_n} \\\\\n"

latex_table += r"""    \bottomrule
  \end{tabular}
\end{subtable}
\end{table}
"""
print("\nLaTeX table (copy-paste to your .tex):\n")
print(latex_table)

# done
print("\nAll done. Figures and LaTeX table printed. If you want, I can also produce a compact CSV export of the ISCO-3 aggregates.")


Loading occ-skills matrix from: /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/processed/esco/occ_skills_matrix.pkl


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/3198198254.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)   # DataFrame rows=occ URIs, cols=skill URIs


occ_skills shape: (3008, 13891) unique values sample: [0 1 2]
Loaded digital/green lists from CSVs: digCompSkillsCollection_en.csv greenSkillsCollection_en.csv
Found digital_uris: 21 Found green_uris: 570
#digital cols present in occ_skills: 21
#green   cols present in occ_skills: 570
Mapped occupations to ISCO-4/ISCO-3 (via esco.occupations).

Compact summary numbers (paste into table):
Digital: #skills, #ever_essential, median_isco4, max4/max3 = 21 9 0 32/13
Green:   #skills, #ever_essential, median_isco4, max4/max3 = 570 473 3 68/23

Top 10 ISCO-3 groups by TOTAL essential GREEN skills (total / n occs-with-any):
 ISCO-3 214 — total essential-green-skills = 555; occupations with any essential-green = 81
 ISCO-3 213 — total essential-green-skills = 253; occupations with any essential-green = 38
 ISCO-3 311 — total essential-green-skills = 244; occupations with any essential-green = 66
 ISCO-3 132 — total essential-green-skills = 136; occupations with any essential-green = 26
 ISCO-3 3

In [13]:
# ---------------------------
# ISCO-3 aggregates diagnostics
# Paste at the bottom of your notebook
# ---------------------------
import pandas as pd
import numpy as np
from pathlib import Path

# assume useful_paths, Esco are already available in this notebook (as you said)
esco = Esco()

# 1) load occ_skills if not in memory
try:
    occ_skills  # noqa: F821
except NameError:
    import pickle, os
    p = Path(useful_paths.data_processed) / "esco" / "occ_skills_matrix.pkl"
    with open(p, "rb") as f:
        occ_skills = pickle.load(f)   # DataFrame rows=occ URIs, cols=skill URIs
print("Loaded occ_skills shape:", occ_skills.shape)

# 2) load digital/green lists if not in memory (you used these before)
try:
    digital_uris, green_uris
except NameError:
    p_dig = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "digCompSkillsCollection_en.csv"
    p_grn = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "greenSkillsCollection_en.csv"
    df_dig = pd.read_csv(p_dig, index_col=0)
    df_grn = pd.read_csv(p_grn, index_col=0)
    digital_uris = df_dig["conceptUri"].tolist()
    green_uris   = df_grn["conceptUri"].tolist()

# 3) get occupation metadata with ISCO grouping and preferredLabel
df_occ = esco.occupations.copy()  # should have columns 'conceptUri' and 'iscoGroup' or 'iscoGroupUri' depending on your Esco
# ensure conceptUri is index or column
if "conceptUri" in df_occ.columns:
    df_occ = df_occ.set_index("conceptUri")
# pick isco value column (esco object varies): try 'iscoGroup' (numeric) or 'iscoGroupUri'
if "iscoGroup" in df_occ.columns:
    isco_col = "iscoGroup"
else:
    # fallback to string-based column name
    isco_col = [c for c in df_occ.columns if "isco" in c.lower()][0]
print("Using occupation ISCO column:", isco_col)

# produce ISCO-3 string (first 3 digits). if numeric like 2654 -> '265'
def to_isco3(x):
    s = str(x)
    # remove non-digits then take first 3
    s2 = "".join([c for c in s if c.isdigit()])
    return s2[:3] if len(s2) >= 3 else np.nan

df_occ["isco3"] = df_occ[isco_col].apply(to_isco3)

# 4) per-occupation essential counts (essential coded as 2 in your matrix)
ess_mask = occ_skills == 2
# restrict to skills we care about (green/digital) OR compute full counts for reference
per_occ_green_ess = ess_mask[green_uris].sum(axis=1) if all(u in occ_skills.columns for u in green_uris) else pd.Series(0, index=occ_skills.index)
per_occ_dig_ess   = ess_mask[digital_uris].sum(axis=1) if all(u in occ_skills.columns for u in digital_uris) else pd.Series(0, index=occ_skills.index)

# 5) join these counts to df_occ (ensure index alignment)
df_occ = df_occ.reindex(occ_skills.index)  # align index (occupation URIs order)
df_occ["green_ess_per_occ"] = per_occ_green_ess
df_occ["dig_ess_per_occ"]   = per_occ_dig_ess

# 6) ISCO-3 aggregations
group = df_occ.groupby("isco3")

# (A) number of occupations in ISCO-3
n_occs = group.size().rename("n_occs")

# (B) mean / median per-occupation essential counts
mean_green_per_occ = group["green_ess_per_occ"].mean().rename("mean_green_per_occ")
median_green_per_occ = group["green_ess_per_occ"].median().rename("median_green_per_occ")
mean_dig_per_occ = group["dig_ess_per_occ"].mean().rename("mean_dig_per_occ")
median_dig_per_occ = group["dig_ess_per_occ"].median().rename("median_dig_per_occ")

# (C) sum of essential occurrences (summing counts across occupations)
sum_green_occ = group["green_ess_per_occ"].sum().rename("sum_green_occ")
sum_dig_occ   = group["dig_ess_per_occ"].sum().rename("sum_dig_occ")

# (D) union distinct essential skills for group: how many distinct green/dig skills are essential in any occupation of the group
# we compute boolean arrays per-occupation for essential and then OR across occupations in group
isco3_groups = {}
for isco3, idx in group.groups.items():
    occ_idx = group.groups[isco3]
    # subset occ_skills for occupations in this group
    sub = occ_skills.loc[occ_idx]
    # distinct essential green skills: any row has value==2 in green_uris
    greens_in_index = [u for u in green_uris if u in sub.columns]
    digs_in_index   = [u for u in digital_uris if u in sub.columns]
    if len(greens_in_index) > 0:
        union_green = (sub[greens_in_index] == 2).any(axis=0).sum()
    else:
        union_green = 0
    if len(digs_in_index) > 0:
        union_dig = (sub[digs_in_index] == 2).any(axis=0).sum()
    else:
        union_dig = 0
    isco3_groups[isco3] = {"union_green": int(union_green), "union_dig": int(union_dig)}

df_union = pd.DataFrame.from_dict(isco3_groups, orient="index").sort_index()

# 7) assemble final ISCO-3 table
isco3_table = pd.concat([n_occs, mean_green_per_occ, median_green_per_occ, sum_green_occ,
                         df_union["union_green"], mean_dig_per_occ, median_dig_per_occ, sum_dig_occ, df_union["union_dig"]], axis=1).fillna(0)
isco3_table = isco3_table.sort_values("union_green", ascending=False)  # sort by green union for inspection

# 8) print the top rows and save CSV
print("\nTop ISCO-3 groups by DISTINCT essential GREEN skills (union):\n")
print(isco3_table.head(12).to_string())

out = Path(useful_paths.output_dir if hasattr(useful_paths, "output_dir") else ".") / "isco3_essential_skill_aggregates.csv"
isco3_table.to_csv(out)
print("\nWrote CSV:", out)

# 9) example: show occupations inside the top ISCO-3 and their per-occ counts (labels)
top_isco3 = isco3_table.index[0]
print("\nTop ISCO-3 group (example):", top_isco3)
sample_occs = df_occ[df_occ["isco3"] == top_isco3].copy()
# map preferredLabel if available
if "preferredLabel" in df_occ.columns:
    sample_occs["label"] = df_occ["preferredLabel"]
else:
    sample_occs["label"] = sample_occs.index
print("\nSample occupations (ISCO-4) in this ISCO-3 group with per-occ essential counts:")
print(sample_occs[["label", "green_ess_per_occ", "dig_ess_per_occ"]].sort_values("green_ess_per_occ", ascending=False).head(20).to_string())


Loaded occ_skills shape: (3008, 13891)
Using occupation ISCO column: iscoGroup

Top ISCO-3 groups by DISTINCT essential GREEN skills (union):

     n_occs  mean_green_per_occ  median_green_per_occ  sum_green_occ  union_green  mean_dig_per_occ  median_dig_per_occ  sum_dig_occ  union_dig
214     125            4.440000                   1.0            555          214          0.016000                 0.0            2          2
213      40            6.325000                   4.0            253          140          0.000000                 0.0            0          0
311     134            1.820896                   0.0            244          129          0.014925                 0.0            2          2
132     118            1.152542                   0.0            136           76          0.000000                 0.0            0          0
313      35            3.657143                   3.0            128           67          0.028571                 0.0            1     

In [14]:
# ---------- paste this at the bottom of your notebook ----------
import os
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from data.framework import Esco

# Paths (adjust only if your useful_paths object is named differently)
p_occ_skills = os.path.join(useful_paths.data_processed, "esco", "occ_skills_matrix.pkl")
p_dig       = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "digCompSkillsCollection_en.csv")
p_grn       = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv")

# 1) load occ--skills matrix
with open(p_occ_skills, "rb") as f:
    occ_skills = pickle.load(f)   # DataFrame: index = occ conceptUri, columns = skill conceptUri

# treat essential as value == 2 (based on your files earlier)
ess_mask = (occ_skills == 2)

# 2) load esco lists
df_dig = pd.read_csv(p_dig, index_col=0)
df_grn = pd.read_csv(p_grn, index_col=0)
digital_uris = df_dig["conceptUri"].tolist()
green_uris   = df_grn["conceptUri"].tolist()

# 3) map occupations to ISCO-3 using Esco() helper
esco = Esco()
# esco.occupations should be a DataFrame with columns including 'conceptUri' and 'iscoGroup' (or 'iscoGroupUri')
df_occ_meta = esco.occupations.copy()
if 'conceptUri' not in df_occ_meta.columns:
    raise RuntimeError("esco.occupations missing conceptUri column")
# create an index by conceptUri
df_occ_meta = df_occ_meta.set_index('conceptUri')

# get an 'iscoGroup' string and derive 3-digit group
# your data has column 'iscoGroup' (e.g. 2654) — convert to string and take first 3 characters
if 'iscoGroup' not in df_occ_meta.columns:
    raise RuntimeError("esco.occupations missing 'iscoGroup' column; please inspect esco.occupations")
df_occ_meta['iscoGroup'] = df_occ_meta['iscoGroup'].astype(str)
df_occ_meta['isco3'] = df_occ_meta['iscoGroup'].str[:3]

# keep only occupations present in occ_skills index
occ_in_matrix = occ_skills.index.intersection(df_occ_meta.index)
print(f"Occs in matrix: {len(occ_in_matrix)} (matching esco.occupations: {len(df_occ_meta)})")
df_occ_meta = df_occ_meta.loc[occ_in_matrix]

# 4) compute per-ISCO-3 union counts (distinct essential skills used anywhere in that ISCO-3)
isco3_groups = df_occ_meta['isco3'].unique()
rows = []
for g in sorted(isco3_groups):
    occs = df_occ_meta.index[df_occ_meta['isco3'] == g].tolist()
    if len(occs) == 0:
        continue
    # restrict to rows in occ_skills
    occs = [o for o in occs if o in occ_skills.index]
    if len(occs) == 0:
        continue
    # union counts for green/digital: is there at least one essential in any occ for that skill?
    u_green = (ess_mask.loc[occs, green_uris].any(axis=0)).sum()
    u_dig   = (ess_mask.loc[occs, digital_uris].any(axis=0)).sum()
    # also record per-occ basic stats if you want
    per_occ_green = ess_mask.loc[occs, green_uris].sum(axis=1)  # count essential green per ISCO-4 occ
    per_occ_dig   = ess_mask.loc[occs, digital_uris].sum(axis=1)
    rows.append({
        "isco3": g,
        "n_occs": len(occs),
        "union_green": int(u_green),
        "union_dig": int(u_dig),
        "mean_green_per_occ": float(per_occ_green.mean()),
        "median_green_per_occ": float(per_occ_green.median()),
        "mean_dig_per_occ": float(per_occ_dig.mean()),
        "median_dig_per_occ": float(per_occ_dig.median())
    })

df_isco3 = pd.DataFrame(rows).set_index("isco3").sort_index()

# 5) compact summary numbers for table
def compact_summary(skill_uris, union_col_name="union_green"):
    n_skills = len(skill_uris)
    ever_essential = int((ess_mask[skill_uris].any(axis=0)).sum())
    median_union = int(df_isco3[union_col_name].median())
    max_union = int(df_isco3[union_col_name].max())
    return n_skills, ever_essential, median_union, max_union

dig_n, dig_ever, dig_med_union, dig_max_union = compact_summary(digital_uris, "union_dig")
grn_n, grn_ever, grn_med_union, grn_max_union = compact_summary(green_uris, "union_green")

print("Digital:", dig_n, dig_ever, dig_med_union, dig_max_union)
print("Green:  ", grn_n, grn_ever, grn_med_union, grn_max_union)

# 6) produce LaTeX table text (pure ISCO-3 metrics)
latex = rf"""
\begin{{table}}[ht]
\centering
\caption{{Essential-skill coverage aggregated to ISCO-3 groups (ESCO lists)}}
\label{{tab:essential_skill_coverage_isco3}}
\begin{{tabular}}{{lrrrr}}
\toprule
               & \# skills in list & \# ever essential (any ISCO-3) & median distinct essential skills per ISCO-3 & max distinct essential skills in any ISCO-3 \\
\midrule
Digital        & {dig_n:>6} & {dig_ever:>6} & {dig_med_union:>10} & {dig_max_union:>10} \\
Green          & {grn_n:>6} & {grn_ever:>6} & {grn_med_union:>10} & {grn_max_union:>10} \\
\bottomrule
\end{{tabular}}
\end{{table}}
"""
print("\nLaTeX table (copy and paste into your .tex):\n")
print(latex)

/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/1764672224.py:16: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)   # DataFrame: index = occ conceptUri, columns = skill conceptUri


Occs in matrix: 3008 (matching esco.occupations: 3008)
Digital: 21 9 0 7
Green:   570 473 6 214

LaTeX table (copy and paste into your .tex):


\begin{table}[ht]
\centering
\caption{Essential-skill coverage aggregated to ISCO-3 groups (ESCO lists)}
\label{tab:essential_skill_coverage_isco3}
\begin{tabular}{lrrrr}
\toprule
               & \# skills in list & \# ever essential (any ISCO-3) & median distinct essential skills per ISCO-3 & max distinct essential skills in any ISCO-3 \\
\midrule
Digital        &     21 &      9 &          0 &          7 \\
Green          &    570 &    473 &          6 &        214 \\
\bottomrule
\end{tabular}
\end{table}



In [15]:
# --------- Paste this at the bottom of your notebook ----------
import os
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
from data.framework import Esco

# ---- paths (use your useful_paths) ----
p_occ_skills = os.path.join(useful_paths.data_processed, "esco", "occ_skills_matrix.pkl")
p_dig       = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "digCompSkillsCollection_en.csv")
p_grn       = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv")

# ---- load occ_skills matrix ----
with open(p_occ_skills, "rb") as f:
    occ_skills = pickle.load(f)   # DataFrame index=occ conceptUri, columns=skill conceptUri

# assume encoding where essential==2 (as in your processed file)
ess_mask = (occ_skills == 2)

# ---- load digital / green lists ----
df_dig = pd.read_csv(p_dig, index_col=0)
df_grn = pd.read_csv(p_grn, index_col=0)
digital_uris = [u for u in df_dig["conceptUri"].tolist() if u in occ_skills.columns]
green_uris   = [u for u in df_grn["conceptUri"].tolist() if u in occ_skills.columns]

# ---- load ESCO occupation metadata and derive ISCO-3 ----
esco = Esco()
df_occ_meta = esco.occupations.copy()  # expected columns include 'conceptUri','preferredLabel','iscoGroup' or 'code'
if 'conceptUri' not in df_occ_meta.columns:
    raise RuntimeError("esco.occupations missing 'conceptUri' column")
df_occ_meta = df_occ_meta.set_index('conceptUri')

# detect the ISCO field name and create iscoGroup string
if 'iscoGroup' in df_occ_meta.columns:
    df_occ_meta['iscoGroup'] = df_occ_meta['iscoGroup'].astype(str)
elif 'code' in df_occ_meta.columns:
    # fallback if occupations have 'code' representing ISCO-like code
    df_occ_meta['iscoGroup'] = df_occ_meta['code'].astype(str)
else:
    raise RuntimeError("esco.occupations has no 'iscoGroup' or 'code' to derive ISCO grouping")

# take first 3 characters as ISCO-3 code (defensive)
df_occ_meta['isco3'] = df_occ_meta['iscoGroup'].str[:3]

# restrict to occupations present in occ_skills
occ_in_matrix = occ_skills.index.intersection(df_occ_meta.index)
df_occ_meta = df_occ_meta.loc[occ_in_matrix]

print(f"Occs in matrix: {len(occ_in_matrix)} (matching esco.occupations: {len(df_occ_meta)})")

# ---- compute ISCO-3 aggregates ----
rows = []
for g, group_df in df_occ_meta.groupby("isco3"):
    occs = group_df.index.tolist()
    n_occs = len(occs)
    # union = distinct skills that are essential for any occ in the group
    union_green = int(ess_mask.loc[occs, green_uris].any(axis=0).sum()) if len(green_uris)>0 else 0
    union_dig   = int(ess_mask.loc[occs, digital_uris].any(axis=0).sum()) if len(digital_uris)>0 else 0
    # sum = aggregate essential counts across all occupations in this group
    sum_green = int(ess_mask.loc[occs, green_uris].sum().sum()) if len(green_uris)>0 else 0
    sum_dig   = int(ess_mask.loc[occs, digital_uris].sum().sum()) if len(digital_uris)>0 else 0
    # per-occ stats
    per_occ_green = ess_mask.loc[occs, green_uris].sum(axis=1) if len(green_uris)>0 else pd.Series(0, index=occs)
    per_occ_dig   = ess_mask.loc[occs, digital_uris].sum(axis=1) if len(digital_uris)>0 else pd.Series(0, index=occs)
    rows.append({
        "isco3": g,
        "n_occs": n_occs,
        "union_green": union_green,
        "union_dig": union_dig,
        "sum_green": sum_green,
        "sum_dig": sum_dig,
        "mean_green_per_occ": float(per_occ_green.mean()),
        "median_green_per_occ": float(per_occ_green.median()),
        "mean_dig_per_occ": float(per_occ_dig.mean()),
        "median_dig_per_occ": float(per_occ_dig.median())
    })

df_isco3 = pd.DataFrame(rows).set_index("isco3")
df_isco3 = df_isco3.sort_values(["union_green","sum_green"], ascending=False)

# ---- compact left-subtable values (ISCO-3 aggregates to paste into LaTeX) ----
def compact_union(skill_uris, union_col):
    n_skills = len(skill_uris)
    ever_essential = int((occ_skills[skill_uris] == 2).any(axis=0).sum()) if n_skills>0 else 0
    median_union = int(df_isco3[union_col].median()) if len(df_isco3)>0 else 0
    max_union = int(df_isco3[union_col].max()) if len(df_isco3)>0 else 0
    return n_skills, ever_essential, median_union, max_union

dig_n, dig_ever, dig_med_union, dig_max_union = compact_union(digital_uris, "union_dig")
grn_n, grn_ever, grn_med_union, grn_max_union = compact_union(green_uris, "union_green")

print("Compact summary numbers (digital / green):")
print("Digital:", dig_n, dig_ever, dig_med_union, dig_max_union)
print("Green:  ", grn_n, grn_ever, grn_med_union, grn_max_union)

# ---- select top-5 ISCO-3 groups (by distinct essential skills = union) ----
top5_isco3_green_by_union = df_isco3.sort_values("union_green", ascending=False).head(5)
top5_isco3_dig_by_union   = df_isco3.sort_values("union_dig", ascending=False).head(5)

print("\nTop 5 ISCO-3 groups by DISTINCT essential GREEN skills (union):")
for idx, r in top5_isco3_green_by_union.iterrows():
    print(f" ISCO-3 {idx} — union_green={r['union_green']}, sum_green={r['sum_green']}, n_occs={r['n_occs']}")

print("\nTop 5 ISCO-3 groups by DISTINCT essential DIGITAL skills (union):")
for idx, r in top5_isco3_dig_by_union.iterrows():
    print(f" ISCO-3 {idx} — union_dig={r['union_dig']}, sum_dig={r['sum_dig']}, n_occs={r['n_occs']}")

# ---- produce LaTeX table with left = ISCO-3 summary; right = top-5 ISCO-3 groups ----
left_subtable = rf"""%
\begin{{tabular}}{{lrrrr}}
\toprule
 & \# skills in list & \# ever essential & median distinct essential skills per ISCO-3 & max distinct essential skills in any ISCO-3 \\
\midrule
Digital & {dig_n} & {dig_ever} & {dig_med_union} & {dig_max_union} \\
Green   & {grn_n} & {grn_ever} & {grn_med_union} & {grn_max_union} \\
\bottomrule
\end{{tabular}}
"""

def make_isco3_top5_table(title, df_top5):
    rows_tex = []
    for i, (idx, row) in enumerate(df_top5.iterrows()):
        rows_tex.append(f"{i+1}. ISCO-3 {idx} & {int(row['union_green'] if 'green' in title.lower() else row['union_dig'])} & {int(row['sum_green'] if 'green' in title.lower() else row['sum_dig'])} & {int(row['n_occs'])} \\\\")
    rows_tex = "\n".join(rows_tex)
    return rf"""\begin{{tabular}}{{lrrr}}
\toprule
\multicolumn{{4}}{{l}}{{\textbf{{{title}}}}} \\
ISCO-3 & \# distinct essential & \# total essential (sum) & \# ISCO-4 occs \\
\midrule
{rows_tex}
\bottomrule
\end{{tabular}}"""

right_subtable = make_isco3_top5_table("Top 5 ISCO-3: essential green skills (distinct / sum / n_occs)", top5_isco3_green_by_union) + "\n\n" + make_isco3_top5_table("Top 5 ISCO-3: essential digital skills (distinct / sum / n_occs)", top5_isco3_dig_by_union)

latex = rf"""
\begin{{table}}[ht]
\centering
\caption{{Essential-skill coverage (ISCO-3 aggregates) and top ISCO-3 groups by essential-skill counts}}
\label{{tab:essential_skill_coverage_and_top_isco3}}
\begin{{subtable}}[t]{{.48\textwidth}}
  \centering
  \caption{{(a) ISCO-3 aggregated skill coverage}}
  {left_subtable}
\end{{subtable}}%
\hfill
\begin{{subtable}}[t]{{.48\textwidth}}
  \centering
  \caption{{(b) Top ISCO-3 groups by essential-skill counts (distinct / sum / n_occs)}}
  {right_subtable}
\end{{subtable}}
\end{{table}}
"""

print("\nLaTeX table (copy-paste into your .tex file):\n")
print(latex)
# ------------------------------------------------------------------


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/160647631.py:16: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)   # DataFrame index=occ conceptUri, columns=skill conceptUri


Occs in matrix: 3008 (matching esco.occupations: 3008)
Compact summary numbers (digital / green):
Digital: 21 9 0 7
Green:   570 473 6 214

Top 5 ISCO-3 groups by DISTINCT essential GREEN skills (union):
 ISCO-3 214 — union_green=214.0, sum_green=555.0, n_occs=125.0
 ISCO-3 213 — union_green=140.0, sum_green=253.0, n_occs=40.0
 ISCO-3 311 — union_green=129.0, sum_green=244.0, n_occs=134.0
 ISCO-3 132 — union_green=76.0, sum_green=136.0, n_occs=118.0
 ISCO-3 313 — union_green=67.0, sum_green=128.0, n_occs=35.0

Top 5 ISCO-3 groups by DISTINCT essential DIGITAL skills (union):
 ISCO-3 221 — union_dig=7.0, sum_dig=13.0, n_occs=2.0
 ISCO-3 214 — union_dig=2.0, sum_dig=2.0, n_occs=125.0
 ISCO-3 311 — union_dig=2.0, sum_dig=2.0, n_occs=134.0
 ISCO-3 722 — union_dig=2.0, sum_dig=3.0, n_occs=46.0
 ISCO-3 818 — union_dig=1.0, sum_dig=1.0, n_occs=24.0

LaTeX table (copy-paste into your .tex file):


\begin{table}[ht]
\centering
\caption{Essential-skill coverage (ISCO-3 aggregates) and top ISCO-3

In [16]:
# ----- Paste this at the bottom of your notebook -----
import os
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
from data.framework import Esco, Classifications

# paths (uses useful_paths you already load earlier in the notebook)
p_occ_skills = os.path.join(useful_paths.data_processed, "esco", "occ_skills_matrix.pkl")
p_dig       = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "digCompSkillsCollection_en.csv")
p_grn       = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv")

# load occ_skills matrix
with open(p_occ_skills, "rb") as f:
    occ_skills = pickle.load(f)   # DataFrame index=occ conceptUri, columns=skill conceptUri

# interpret essential mask (your processed matrix uses integers: essential==2)
ess_mask = (occ_skills == 2)

# load digital / green lists
df_dig = pd.read_csv(p_dig, index_col=0)
df_grn = pd.read_csv(p_grn, index_col=0)
digital_uris = [u for u in df_dig["conceptUri"].tolist() if u in occ_skills.columns]
green_uris   = [u for u in df_grn["conceptUri"].tolist() if u in occ_skills.columns]

# load ESCO occupations metadata to get iscoGroup
esco = Esco()
df_occ_meta = esco.occupations.copy()  # expected columns include 'conceptUri' and an ISCO grouping field
if 'conceptUri' not in df_occ_meta.columns:
    raise RuntimeError("esco.occupations missing 'conceptUri' column")
df_occ_meta = df_occ_meta.set_index('conceptUri')

# detect ISCO group field (defensive)
if 'iscoGroup' in df_occ_meta.columns:
    df_occ_meta['iscoGroup_str'] = df_occ_meta['iscoGroup'].astype(str)
elif 'code' in df_occ_meta.columns:
    df_occ_meta['iscoGroup_str'] = df_occ_meta['code'].astype(str)
elif 'iscoGroupUri' in df_occ_meta.columns:
    # sometimes there is an URI like ".../ISCO08_214", try to parse the trailing digits
    df_occ_meta['iscoGroup_str'] = df_occ_meta['iscoGroupUri'].str.extract(r'(\d{3})$')[0].fillna(df_occ_meta['iscoGroupUri'])
else:
    raise RuntimeError("esco.occupations has no obvious ISCO field ('iscoGroup' / 'code' / 'iscoGroupUri')")

# canonical ISCO-3 code = first 3 characters (defensive trimming)
df_occ_meta['isco3'] = df_occ_meta['iscoGroup_str'].str[:3]

# restrict to occupations present in occ_skills
occ_in_matrix = occ_skills.index.intersection(df_occ_meta.index)
df_occ_meta = df_occ_meta.loc[occ_in_matrix]

print(f"Occs in matrix: {len(occ_in_matrix)} (matching esco.occupations: {len(df_occ_meta)})")

# ---- compute ISCO-3 aggregates ----
rows = []
for g, group_df in df_occ_meta.groupby("isco3"):
    occs = group_df.index.tolist()
    n_occs = len(occs)
    # distinct (union) essential skills in the ISCO-3 group
    union_green = int(ess_mask.loc[occs, green_uris].any(axis=0).sum()) if len(green_uris) else 0
    union_dig   = int(ess_mask.loc[occs, digital_uris].any(axis=0).sum()) if len(digital_uris) else 0
    # total essential skill counts across all occs in the group
    sum_green = int(ess_mask.loc[occs, green_uris].sum().sum()) if len(green_uris) else 0
    sum_dig   = int(ess_mask.loc[occs, digital_uris].sum().sum()) if len(digital_uris) else 0
    # per-occ stats
    per_occ_green = ess_mask.loc[occs, green_uris].sum(axis=1) if len(green_uris) else pd.Series(0, index=occs)
    per_occ_dig   = ess_mask.loc[occs, digital_uris].sum(axis=1) if len(digital_uris) else pd.Series(0, index=occs)
    rows.append({
        "isco3": g,
        "n_occs": n_occs,
        "union_green": union_green,
        "union_dig": union_dig,
        "sum_green": sum_green,
        "sum_dig": sum_dig,
        "mean_green_per_occ": float(per_occ_green.mean()),
        "median_green_per_occ": float(per_occ_green.median()),
        "mean_dig_per_occ": float(per_occ_dig.mean()),
        "median_dig_per_occ": float(per_occ_dig.median())
    })

df_isco3 = pd.DataFrame(rows).set_index("isco3")

# ---- try to obtain ISCO-3 human-readable labels (multiple fallbacks) ----
isco_label_map = {}

# 1) Try using Classifications object (if it exposes ISCO entries)
try:
    cls = Classifications()
    # common possible attributes / methods, try a few safe ones
    if hasattr(cls, "isco") and isinstance(cls.isco, (pd.DataFrame, dict)):
        if isinstance(cls.isco, pd.DataFrame):
            if 'code' in cls.isco.columns and 'label' in cls.isco.columns:
                isco_label_map = cls.isco.set_index('code')['label'].to_dict()
        else:
            isco_label_map = dict(cls.isco)
    elif hasattr(cls, "get") and callable(cls.get):
        # some implementations: cls.get("ISCO") or cls.get("ISCO08")
        for key in ("ISCO","ISCO08","isco","isco08"):
            try:
                res = cls.get(key)
                if isinstance(res, pd.DataFrame) and 'code' in res.columns and 'label' in res.columns:
                    isco_label_map = res.set_index('code')['label'].to_dict()
                    break
            except Exception:
                continue
except Exception:
    isco_label_map = {}

# 2) Try reading common CSV locations in the repo (useful_paths.data_raw/classifications ...)
if not isco_label_map:
    candidate_paths = [
        os.path.join(useful_paths.data_raw, "classifications", "ISCO08.csv"),
        os.path.join(useful_paths.data_raw, "classifications", "isco08.csv"),
        os.path.join(useful_paths.data_raw, "classifications", "ISCO.csv"),
        os.path.join(useful_paths.data_raw, "isco08.csv"),
        os.path.join(useful_paths.data_raw, "ISCO08.csv"),
    ]
    for p in candidate_paths:
        if os.path.exists(p):
            try:
                df = pd.read_csv(p)
                # try to find plausible columns
                if 'code' in df.columns and 'label' in df.columns:
                    isco_label_map = df.set_index('code')['label'].astype(str).to_dict()
                    break
                # try other heuristics
                for c_code in ['code','Code','ISCO_CODE','isco','id']:
                    for c_label in ['label','Label','description','Description','name','Name']:
                        if c_code in df.columns and c_label in df.columns:
                            isco_label_map = df.set_index(c_code)[c_label].astype(str).to_dict()
                            break
                    if isco_label_map:
                        break
            except Exception:
                continue

# 3) final fallback: try to build simple label from occupations metadata (take most common preferredLabel pattern in the ISCO-3 group)
if not isco_label_map:
    # build a minimal mapping by using the first few occupation labels in each isco3 group
    try:
        if 'preferredLabel' in df_occ_meta.columns:
            for code, group in df_occ_meta.groupby('isco3'):
                # get a compact string like "Energy & related engineers (sample occ: energy engineer)"
                sample_label = group['preferredLabel'].iloc[0]
                isco_label_map[code] = f"ISCO-3 {code} ({sample_label})"
    except Exception:
        isco_label_map = {}

# if still empty, warn and leave numeric codes as labels
if not isco_label_map:
    print("WARNING: could not find a local ISCO label mapping. The output will use ISCO numeric codes. "
          "If you have a local ISCO mapping CSV, place it under data/raw/classifications/ISCO08.csv with columns 'code' and 'label' and re-run.")
    # create simple identity map so code appears as "ISCO-3 214"
    isco_label_map = {code: f"ISCO-3 {code}" for code in df_isco3.index.tolist()}

# now create a label column in df_isco3
df_isco3['isco3_label'] = df_isco3.index.map(lambda c: isco_label_map.get(str(c), f"ISCO-3 {c}"))

# ---- compact left-subtable values (the numbers you asked to include in the LaTeX) ----
def compact_union(skill_uris, union_col):
    n_skills = len(skill_uris)
    ever_essential = int((occ_skills[skill_uris] == 2).any(axis=0).sum()) if n_skills>0 else 0
    median_union = int(df_isco3[union_col].median()) if len(df_isco3)>0 else 0
    max_union = int(df_isco3[union_col].max()) if len(df_isco3)>0 else 0
    return n_skills, ever_essential, median_union, max_union

dig_n, dig_ever, dig_med_union, dig_max_union = compact_union(digital_uris, "union_dig")
grn_n, grn_ever, grn_med_union, grn_max_union = compact_union(green_uris, "union_green")

print("Compact summary numbers (digital / green):")
print("Digital:", dig_n, dig_ever, dig_med_union, dig_max_union)
print("Green:  ", grn_n, grn_ever, grn_med_union, grn_max_union)

# ---- top-5 ISCO-3 by distinct essential skills (union) ----
top5_isco3_green_by_union = df_isco3.sort_values("union_green", ascending=False).head(5)
top5_isco3_dig_by_union   = df_isco3.sort_values("union_dig", ascending=False).head(5)

print("\nTop 5 ISCO-3 groups by DISTINCT essential GREEN skills (union) — labels shown:")
for idx, r in top5_isco3_green_by_union.iterrows():
    print(f" {r['isco3_label']} — distinct essential green={int(r['union_green'])}, total essential (sum)={int(r['sum_green'])}, n_ISCO4_occs={int(r['n_occs'])}")

print("\nTop 5 ISCO-3 groups by DISTINCT essential DIGITAL skills (union) — labels shown:")
for idx, r in top5_isco3_dig_by_union.iterrows():
    print(f" {r['isco3_label']} — distinct essential digital={int(r['union_dig'])}, total essential (sum)={int(r['sum_dig'])}, n_ISCO4_occs={int(r['n_occs'])}")

# ---- Build LaTeX table text with labels ----
left_subtable = rf"""%
\begin{{tabular}}{{lrrrr}}
\toprule
 & \# skills in list & \# ever essential & median distinct essential skills per ISCO-3 & max distinct essential skills in any ISCO-3 \\
\midrule
Digital & {dig_n} & {dig_ever} & {dig_med_union} & {dig_max_union} \\
Green   & {grn_n} & {grn_ever} & {grn_med_union} & {grn_max_union} \\
\bottomrule
\end{{tabular}}
"""

def make_isco3_top5_table_labeled(title, df_top5, kind="green"):
    rows_tex = []
    for i, (idx, row) in enumerate(df_top5.iterrows()):
        label = row['isco3_label']
        distinct = int(row['union_green'] if kind=="green" else row['union_dig'])
        total = int(row['sum_green'] if kind=="green" else row['sum_dig'])
        n_occs = int(row['n_occs'])
        # escape underscores in labels for LaTeX
        label_tex = label.replace('_','\\_')
        rows_tex.append(f"{label_tex} & {distinct} & {total} & {n_occs} \\\\")
    rows_tex = "\n".join(rows_tex)
    return rf"""\begin{{tabular}}{{lrrr}}
\toprule
\multicolumn{{4}}{{l}}{{\textbf{{{title}}}}} \\
ISCO-3 group & \# distinct essential & \# total essential (sum) & \# ISCO-4 occs \\
\midrule
{rows_tex}
\bottomrule
\end{{tabular}}"""

right_subtable = make_isco3_top5_table_labeled("Top 5 ISCO-3: essential green skills (distinct / sum / n_occs)", top5_isco3_green_by_union, kind="green") + "\n\n" + make_isco3_top5_table_labeled("Top 5 ISCO-3: essential digital skills (distinct / sum / n_occs)", top5_isco3_dig_by_union, kind="dig")

latex = rf"""
\begin{{table}}[ht]
\centering
\caption{{Essential-skill coverage (ISCO-3 aggregates) and top ISCO-3 groups by essential-skill counts (labels shown)}}
\label{{tab:essential_skill_coverage_and_top_isco3_labeled}}
\begin{{subtable}}[t]{{.48\textwidth}}
  \centering
  \caption{{(a) ISCO-3 aggregated skill coverage}}
  {left_subtable}
\end{{subtable}}%
\hfill
\begin{{subtable}}[t]{{.48\textwidth}}
  \centering
  \caption{{(b) Top ISCO-3 groups by essential-skill counts (distinct / sum / n_occs)}}
  {right_subtable}
\end{{subtable}}
\end{{table}}
"""

print("\nLaTeX table (copy-paste into your .tex file):\n")
print(latex)

/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/2918317721.py:16: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)   # DataFrame index=occ conceptUri, columns=skill conceptUri


Occs in matrix: 3008 (matching esco.occupations: 3008)
Compact summary numbers (digital / green):
Digital: 21 9 0 7
Green:   570 473 6 214

Top 5 ISCO-3 groups by DISTINCT essential GREEN skills (union) — labels shown:
 ISCO-3 214 (dismantling engineer) — distinct essential green=214, total essential (sum)=555, n_ISCO4_occs=125
 ISCO-3 213 (pharmacologist) — distinct essential green=140, total essential (sum)=253, n_ISCO4_occs=40
 ISCO-3 311 (asphalt laboratory technician) — distinct essential green=129, total essential (sum)=244, n_ISCO4_occs=134
 ISCO-3 132 (import export manager in meat and meat products) — distinct essential green=76, total essential (sum)=136, n_ISCO4_occs=118
 ISCO-3 313 (petroleum pump system operator) — distinct essential green=67, total essential (sum)=128, n_ISCO4_occs=35

Top 5 ISCO-3 groups by DISTINCT essential DIGITAL skills (union) — labels shown:
 ISCO-3 221 (general practitioner) — distinct essential digital=7, total essential (sum)=13, n_ISCO4_occs=2


In [21]:
# --- Paste this at the end of your notebook ---
import os
import re
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

# --- use existing useful_paths/esco objects in your session ---
try:
    useful_paths
except NameError:
    # try to recreate the usual setup (same as you do in other cells)
    import sys
    project_root = Path().resolve().parents[0]
    sys.path.append(str(project_root / "src"))
    from src import utils
    useful_paths = utils.UsefulPaths(fn_config_path="paths_config.yml")

# try to get Esco object if available (for isco group labels)
try:
    from data.framework import Esco
    esco = Esco()
except Exception:
    esco = None

# 1) load occ_skills
p_occ_skills = os.path.join(useful_paths.data_processed, "esco", "occ_skills_matrix.pkl")
print("Loading occ-skills matrix from:", p_occ_skills)
with open(p_occ_skills, "rb") as f:
    occ_skills = pickle.load(f)   # DataFrame indexed by occupation conceptUri, columns = skill conceptUri
print("occ_skills shape:", occ_skills.shape)
uniq = np.unique(occ_skills.values)
nonzero = uniq[uniq != 0]
if len(nonzero) == 0:
    raise RuntimeError("occ_skills contains only zeros")
essential_code = int(max(nonzero))
optional_code = int(min(nonzero)) if len(nonzero) > 1 else None
print("Unique values (sample):", uniq[:10], "-> essential_code =", essential_code, "optional_code =", optional_code)

# 2) load digital / green lists
p_dig = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "digCompSkillsCollection_en.csv")
p_grn = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "greenSkillsCollection_en.csv")
df_dig = pd.read_csv(p_dig, index_col=0)
df_grn = pd.read_csv(p_grn, index_col=0)
digital_uris = df_dig["conceptUri"].tolist()
green_uris = df_grn["conceptUri"].tolist()
dig_cols = [u for u in digital_uris if u in occ_skills.columns]
grn_cols = [u for u in green_uris if u in occ_skills.columns]
print("Found digital_uris:", len(digital_uris), "present in matrix:", len(dig_cols))
print("Found green_uris:", len(green_uris), "present in matrix:", len(grn_cols))

# 3) load occupation metadata (prefer esco.occupations)
occ_meta = None
if esco is not None and hasattr(esco, "occupations"):
    occ_meta = esco.occupations.copy()
else:
    p_occs_csv = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "occupationsCollection_en.csv")
    if os.path.exists(p_occs_csv):
        occ_meta = pd.read_csv(p_occs_csv, index_col=0)
    else:
        raise RuntimeError("No occupation metadata available (esco object or occupationsCollection_en.csv).")

# ensure index is conceptUri
if "conceptUri" in occ_meta.columns:
    occ_meta = occ_meta.set_index("conceptUri", drop=False)

# restrict to occs present in occ_skills
occ_meta = occ_meta.loc[occ_meta.index.intersection(occ_skills.index)].copy()
print("Occs in meta intersecting matrix:", len(occ_meta), "Occs in matrix:", occ_skills.shape[0])

# 4) extract ISCO-3 code *reliably* from the metadata 'iscoGroup' (or 'code' if needed)
def extract_isco3_from_value(v):
    if pd.isna(v):
        return None
    s = str(v)
    # prefer a plain 4-digit number -> take first 3 digits
    m4 = re.search(r"\b(\d{4})\b", s)
    if m4:
        return m4.group(1)[:3]
    # find any 3-digit group
    m3 = re.search(r"\b(\d{3})\b", s)
    if m3:
        return m3.group(1)
    # last-digit sequence
    m_end = re.search(r"(\d{3})\D*$", s)
    if m_end:
        return m_end.group(1)
    return None

# try common columns in order
candidate_cols = [c for c in ["iscoGroup", "iscoGroupUri", "code", "isco_group"] if c in occ_meta.columns]
if not candidate_cols:
    candidate_cols = [c for c in occ_meta.columns if "isco" in c.lower() or "code" in c.lower()]

isco3_map = {}
for uri, row in occ_meta.iterrows():
    found = None
    for c in candidate_cols:
        if c in row and not pd.isna(row[c]):
            found = extract_isco3_from_value(row[c])
            if found:
                break
    # final fallback: try to extract from 'conceptUri'
    if found is None:
        found = extract_isco3_from_value(uri)
    isco3_map[uri] = found if found is not None else "UNK"

# group occs by isco3 (only those present in occ_skills)
isco3_to_occs = defaultdict(list)
for occ_uri, code in isco3_map.items():
    if occ_uri in occ_skills.index:
        isco3_to_occs[code].append(occ_uri)
print("Detected ISCO-3 groups:", len(isco3_to_occs))

# 5) try to get ISCO-3 human labels (prefer esco.isco_groups)
isco3_label_map = {}
if esco is not None and hasattr(esco, "isco_groups"):
    try:
        ig = esco.isco_groups
        # ig expected to have 'code' and 'preferredLabel'
        if "code" in ig.columns and "preferredLabel" in ig.columns:
            isco3_label_map = ig.set_index("code")["preferredLabel"].to_dict()
    except Exception:
        isco3_label_map = {}

# fallback: try to load a local isco groups CSV
if not isco3_label_map:
    p_isco_csv = os.path.join(useful_paths.data_raw, "esco", "v1.1.0", "iscoGroups_en.csv")
    if os.path.exists(p_isco_csv):
        try:
            df_isco = pd.read_csv(p_isco_csv, index_col=0)
            if "code" in df_isco.columns and "preferredLabel" in df_isco.columns:
                isco3_label_map = df_isco.set_index("code")["preferredLabel"].to_dict()
        except Exception:
            pass

# final fallback: label = "ISCO-3 XXX"
for code in list(isco3_to_occs.keys()):
    if code not in isco3_label_map:
        isco3_label_map[code] = f"ISCO-3 {code}"

# 6) aggregate at ISCO-3 level (distinct essential skill counts)
rows = []
for isco3_code, occ_list in sorted(isco3_to_occs.items(), key=lambda x: (str(x[0]))):
    occs_present = [o for o in occ_list if o in occ_skills.index]
    n_isco4 = len(occs_present)
    if n_isco4 == 0:
        n_green = 0
        n_dig = 0
    else:
        sub = occ_skills.loc[occs_present]
        n_green = int((sub[grn_cols] == essential_code).any(axis=0).sum()) if grn_cols else 0
        n_dig   = int((sub[dig_cols] == essential_code).any(axis=0).sum()) if dig_cols else 0
    rows.append((isco3_code, isco3_label_map.get(isco3_code, f"ISCO-3 {isco3_code}"), n_green, n_dig, n_isco4))

df_isco3 = pd.DataFrame(rows, columns=["isco3_code","isco3_label","n_green_distinct","n_dig_distinct","n_isco4_occs"])

# 7) compute table (a) numbers at ISCO-3 level
n_dig_skills = len(dig_cols)
n_grn_skills = len(grn_cols)

# For "# ever essential (ISCO-3)" count how many skills are essential in at least 1 ISCO-3 group
def ever_essential_isco3(skill_list):
    cnt = 0
    counts = []
    for s in skill_list:
        # count ISCO-3 groups where skill s appears as essential in at least one occ
        c = 0
        for isco3_code, occs in isco3_to_occs.items():
            occs_present = [o for o in occs if o in occ_skills.index]
            if not occs_present:
                continue
            if (occ_skills.loc[occs_present, s] == essential_code).any():
                c += 1
        counts.append(c)
        if c > 0:
            cnt += 1
    return cnt, counts

ever_dig, dig_counts = ever_essential_isco3(dig_cols) if n_dig_skills>0 else (0, [])
ever_grn, grn_counts = ever_essential_isco3(grn_cols) if n_grn_skills>0 else (0, [])

median_dig = int(np.median(dig_counts)) if dig_counts else 0
median_grn = int(np.median(grn_counts)) if grn_counts else 0

df_a = pd.DataFrame({
    "# skills": [n_dig_skills, n_grn_skills],
    "# ever essential (ISCO-3)": [ever_dig, ever_grn],
    "median # occs": [median_dig, median_grn]
}, index=["Digital","Green"])

print("\n=== (a) ISCO-3 aggregated coverage ===")
print(df_a.to_string())

# 8) Top 5 ISCO-3 groups by distinct essential skill counts (table b)
top5_green = df_isco3.sort_values("n_green_distinct", ascending=False).head(5)
top5_dig   = df_isco3.sort_values("n_dig_distinct", ascending=False).head(5)

print("\n=== (b) Top 5 ISCO-3 by DISTINCT essential GREEN skills ===")
print(top5_green[["isco3_label","n_green_distinct","n_dig_distinct","n_isco4_occs"]].to_string(index=False))

print("\n=== (b) Top 5 ISCO-3 by DISTINCT essential DIGITAL skills ===")
print(top5_dig[["isco3_label","n_green_distinct","n_dig_distinct","n_isco4_occs"]].to_string(index=False))

# show a small sample of underlying ISCO-4 occupations for the top green ISCO-3 (sanity check)
example_isco3 = top5_green.iloc[0]["isco3_code"]
sample_occs = isco3_to_occs[example_isco3][:10]
print("\nSample ISCO-4 occupations for top ISCO-3 ({}):".format(top5_green.iloc[0]["isco3_label"]))
print(occ_meta.loc[sample_occs, ["preferredLabel","iscoGroup"]].head(10).to_string())

# 9) print LaTeX for table (a) + table (b) (3-digit labels)
latex = r"""
\begin{table}[ht]
\centering
\caption{Essential-skill coverage aggregated to ISCO-3 groups (ESCO lists)}
\label{tab:essential_skill_coverage_isco3}
\begin{subtable}[t]{.48\textwidth}
  \centering
  \caption{(a) ISCO-3 aggregated skill coverage}
  \begin{tabular}{lrrr}
    \toprule
      & \# skills & \# ever essential (ISCO-3) & median \# occs \\
    \midrule
    Digital & %d & %d & %d \\
    Green   & %d & %d & %d \\
    \bottomrule
  \end{tabular}
\end{subtable}%%
\hfill
\begin{subtable}[t]{.48\textwidth}
  \centering
  \caption{(b) Top ISCO-3 groups by distinct essential-skill counts}
  \begin{tabular}{lrrr}
    \toprule
    Occupation (ISCO-3) & \# green skills & \# digital skills & \# ISCO-4 occs \\
    \midrule
""" % (n_dig_skills, ever_dig, median_dig, n_grn_skills, ever_grn, median_grn)

for _, r in top5_green.iterrows():
    latex += "    %s & %d & %d & %d \\\\\n" % (r['isco3_label'].replace("&","\\&"), int(r['n_green_distinct']), int(r['n_dig_distinct']), int(r['n_isco4_occs']))
latex += "    \\midrule\n"
for _, r in top5_dig.iterrows():
    latex += "    %s & %d & %d & %d \\\\\n" % (r['isco3_label'].replace("&","\\&"), int(r['n_green_distinct']), int(r['n_dig_distinct']), int(r['n_isco4_occs']))
latex += r"""    \bottomrule
  \end{tabular}
\end{subtable}
\end{table}
"""
print("\n--- LaTeX table (copy-paste) ---\n")
print(latex)

Loading occ-skills matrix from: /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/processed/esco/occ_skills_matrix.pkl
occ_skills shape: (3008, 13891)


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/2977577312.py:33: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)   # DataFrame indexed by occupation conceptUri, columns = skill conceptUri


Unique values (sample): [0 1 2] -> essential_code = 2 optional_code = 1
Found digital_uris: 21 present in matrix: 21
Found green_uris: 570 present in matrix: 570
Occs in meta intersecting matrix: 3008 Occs in matrix: 3008
Detected ISCO-3 groups: 125

=== (a) ISCO-3 aggregated coverage ===
         # skills  # ever essential (ISCO-3)  median # occs
Digital        21                          9              0
Green         570                        473              2

=== (b) Top 5 ISCO-3 by DISTINCT essential GREEN skills ===
                                                   isco3_label  n_green_distinct  n_dig_distinct  n_isco4_occs
       Engineering professionals (excluding electrotechnology)               214               2           125
                                    Life science professionals               140               0            40
                  Physical and engineering science technicians               129               2           134
Manufacturing, mining, co

In [24]:
# Diagnostic chunk to find exactly which digital skills are considered essential for the "medical doctors" ISCO-3 group
import numpy as np
import pandas as pd
from textwrap import shorten

# -------------------- Defensive pre-checks --------------------
assert 'occ_skills' in globals(), "occ_skills DataFrame not found. Load occ_skills first."
assert 'esco' in globals(), "esco object not found. Load Esco() as `esco`."

# detect available skill-label maps (prefer df_coreness if present)
skill_label_map = None
if 'df_coreness' in globals():
    skill_label_map = df_coreness.set_index('conceptUri')['preferredLabel'].to_dict()
else:
    # try ESCO skills
    if hasattr(esco, 'skills') and 'conceptUri' in esco.skills.columns and 'preferredLabel' in esco.skills.columns:
        skill_label_map = esco.skills.set_index('conceptUri')['preferredLabel'].to_dict()
    else:
        # fallback: use df_dig and df_grn if present
        tmp = {}
        if 'df_dig' in globals() and 'conceptUri' in df_dig.columns:
            tmp.update(df_dig.set_index('conceptUri').get('preferredLabel', pd.Series()).to_dict())
        if 'df_grn' in globals() and 'conceptUri' in df_grn.columns:
            tmp.update(df_grn.set_index('conceptUri').get('preferredLabel', pd.Series()).to_dict())
        skill_label_map = tmp

def label_for(uri):
    return skill_label_map.get(uri, uri)

# -------------------- detect essential/optional codes --------------------
vals = np.unique(occ_skills.values.ravel())
vals = [v for v in vals if not (pd.isna(v))]
vals_sorted = sorted(vals)
print("Unique values in occ_skills (sample):", vals_sorted)
if set([0,1,2]).issubset(set(vals_sorted)) or (2 in vals_sorted and 1 in vals_sorted):
    # convention we saw: essential->2 optional->1
    essential_code = 2
    optional_code  = 1
elif set([0,1]).issubset(set(vals_sorted)) and 2 not in vals_sorted:
    # only binary: treat 1 as essential
    essential_code = 1
    optional_code  = None
else:
    # best effort fallback
    essential_code = max(vals_sorted)
    optional_code = None if len(vals_sorted)==2 else min([v for v in vals_sorted if v not in (0, essential_code)])

print(f"Detected essential_code = {essential_code}, optional_code = {optional_code}")

# -------------------- locate ISCO-3 code for medical doctors --------------------
# try to find the ISCO-3 code by searching labels
occ_meta = esco.occupations.copy()
if 'conceptUri' in occ_meta.columns:
    occ_meta = occ_meta.set_index('conceptUri')

# check what column holds ISCO group
if 'iscoGroup' not in occ_meta.columns and 'code' in occ_meta.columns:
    # fallback: try 'code' or 'isco' columns
    print("Warning: 'iscoGroup' not found in esco.occupations; columns:", occ_meta.columns.tolist())

# Build isco string per occ in occ_skills index
occ_meta_sub = occ_meta.reindex(occ_skills.index)
isco_raw = occ_meta_sub.get('iscoGroup', occ_meta_sub.get('code')).astype(str).fillna('NA')
isco3 = isco_raw.str[:3]

# Try to find code(s) for medical doctors by matching label keywords
candidates = occ_meta_sub['preferredLabel'].dropna().reset_index()
mask = candidates['preferredLabel'].str.lower().str.contains('medical doctor|general practitioner|physician|specialised doctor', na=False)
cands = candidates[mask]
if len(cands) == 0:
    print("No direct 'medical doctor' label matched in esco.occupations. You may want to specify the ISCO-3 code manually.")
    print("Sample labels (first 20) to inspect:", candidates['preferredLabel'].unique()[:20])
else:
    print("Sample matched occupation labels for doctors (showing up to 10):")
    display(cands.head(10))

# Let user pick ISCO-3 code - default try '221' because that's usual for medical doctors (ISCO-08)
target_isco3 = '221'
# if the search above found something, override:
if not cands.empty:
    target_isco3 = str(isco_raw.loc[cands['conceptUri'].iloc[0]])[:3]

print("Using ISCO-3 code for diagnostics =", target_isco3)

# -------------------- occupations in that ISCO-3 --------------------
occ_in_isco3 = occ_meta_sub[isco3 == target_isco3].index.tolist()
print(f"\nNumber of ISCO-4 occupations inside ISCO-3 {target_isco3}: {len(occ_in_isco3)}")
print("Example ISCO-4 occupations (label, conceptUri, iscoGroup):")
for uri in occ_in_isco3[:40]:
    lbl = occ_meta_sub.loc[uri, 'preferredLabel'] if pd.notna(occ_meta_sub.loc[uri, 'preferredLabel']) else uri
    print(" -", lbl, "|", uri, "| iscoGroup:", occ_meta_sub.loc[uri, 'iscoGroup'])

# -------------------- digital skill columns present --------------------
if 'digital_uris' in globals():
    dig_cols = [u for u in digital_uris if u in occ_skills.columns]
else:
    # try to infer digital URIs using df_dig if present
    dig_cols = []
    if 'df_dig' in globals() and 'conceptUri' in df_dig.columns:
        dig_cols = [u for u in df_dig['conceptUri'].tolist() if u in occ_skills.columns]

print(f"\nFound digital URIs: {len(dig_cols)} (present in occ_skills)")

# -------------------- per-occupation digital essential/optional breakdown --------------------
rows = []
for uri in occ_in_isco3:
    lbl = occ_meta_sub.loc[uri, 'preferredLabel'] if pd.notna(occ_meta_sub.loc[uri, 'preferredLabel']) else uri
    # boolean mask of essential digital skills for this occ:
    if len(dig_cols) == 0:
        ess_digs = []
        opt_digs = []
    else:
        rowvals = occ_skills.loc[uri, dig_cols]
        ess_digs = [c for c,v in rowvals.items() if v == essential_code]
        opt_digs = [c for c,v in rowvals.items() if optional_code is not None and v == optional_code]
    rows.append((uri, lbl, len(ess_digs), len(opt_digs), ess_digs[:20], opt_digs[:20]))

# print neatly
print("\nPer-occupation digital counts (ISCO-4 occupations inside ISCO-3):")
for uri,lbl,n_ess,n_opt,ess_sample,opt_sample in rows:
    if n_ess>0 or n_opt>0:
        ess_labels = [shorten(label_for(u), width=40) for u in ess_sample]
        opt_labels = [shorten(label_for(u), width=40) for u in opt_sample]
        print(f" - {lbl} | essential:{n_ess} optional:{n_opt}")
        if n_ess>0:
            print("    essential skills (sample):", "; ".join(ess_labels))
        if n_opt>0:
            print("    optional skills (sample):", "; ".join(opt_labels))

# -------------------- union (distinct) digital skills across the ISCO-3 --------------------
all_ess_uris = set()
all_opt_uris = set()
for uri,lbl,n_ess,n_opt,ess_sample,opt_sample in rows:
    # gather across full per-occ rows (not just sample)
    if len(dig_cols)>0:
        full_row = occ_skills.loc[uri, dig_cols]
        all_ess_uris.update([c for c,v in full_row.items() if v == essential_code])
        if optional_code is not None:
            all_opt_uris.update([c for c,v in full_row.items() if v == optional_code])

print(f"\nDistinct digital skill URIs essential in ISCO-3 {target_isco3}: {len(all_ess_uris)}")
if len(all_ess_uris)>0:
    print("List of distinct essential digital skill labels (up to 30):")
    for u in list(all_ess_uris)[:30]:
        print(" *", label_for(u), "|", u)

# -------------------- sanity checks: are the URIs present in df_dig? --------------------
if 'df_dig' in globals():
    df_dig_uris = set(df_dig['conceptUri'].tolist())
    in_list = [u for u in all_ess_uris if u in df_dig_uris]
    not_in_list = [u for u in all_ess_uris if u not in df_dig_uris]
    print(f"\nSanity: of {len(all_ess_uris)} distinct essential digital URIs, {len(in_list)} are in df_dig, {len(not_in_list)} are not.")
    if len(not_in_list)>0:
        print("Sample URIs not in your df_dig (up to 10):")
        for u in not_in_list[:10]:
            print(" -", u, "label:", label_for(u))
else:
    print("\nNo df_dig available to check membership of URIs.")

# -------------------- summary hints --------------------
print("\n--- Quick interpretation hints ---")
print(" - If you see essential digital URIs here, they come directly from the occ_skills matrix cells == essential_code.")
print(" - If you believe ESCO website shows NO digital requirements for doctors, possible causes:")
print("   (a) the occ_skills matrix you loaded is NOT the original ESCO file but a processed version that propagated skills up/down ISCO levels or merged lists;")
print("   (b) you are looking at a different ESCO VERSION than the website snapshot (version mismatch);")
print("   (c) essential_code interpretation is swapped (we attempted to auto-detect it above). Check the 'essential_code' printed earlier.")
print(" - Next steps I recommend after you inspect results above:")
print("   1) Inspect a couple of occ URIs that show digital essential (print occ page in ESCO or open occupationsCollection_en.csv to confirm);")
print("   2) Check the exact CSV used for digital list (df_dig), ensure it is v1.1.0 and matches the website snapshot you compare to;")
print("   3) If occ_skills was produced by an earlier script, inspect that script for 'propagate_up' or 'propagate_to_parents' steps that would copy skills from sub-occupations up into parents.")

# End of diagnostic


Unique values in occ_skills (sample): [np.int64(0), np.int64(1), np.int64(2)]
Detected essential_code = 2, optional_code = 1
Sample matched occupation labels for doctors (showing up to 10):


,conceptUri,preferredLabel
1390,http://data.europa.eu/esco/occupation/71f09c8b...,general practitioner
1904,http://data.europa.eu/esco/occupation/9b889f07...,specialised doctor


Using ISCO-3 code for diagnostics = 221

Number of ISCO-4 occupations inside ISCO-3 221: 2
Example ISCO-4 occupations (label, conceptUri, iscoGroup):
 - general practitioner | http://data.europa.eu/esco/occupation/71f09c8b-a172-408c-b9e7-32e580e39ff6 | iscoGroup: 2211
 - specialised doctor | http://data.europa.eu/esco/occupation/9b889f07-c39c-464d-b9d9-b2daa650f9ac | iscoGroup: 2212

Found digital URIs: 21 (present in occ_skills)

Per-occupation digital counts (ISCO-4 occupations inside ISCO-3):
 - general practitioner | essential:7 optional:0
    essential skills (sample): evaluate data, information and [...]; identify digital competence gaps; protect personal data and privacy; manage data, information and [...]; interact through digital technologies; protect health and well-being [...]; browse, search and filter data, [...]
 - specialised doctor | essential:6 optional:0
    essential skills (sample): evaluate data, information and [...]; identify digital competence gaps; protect pers

In [25]:
import os
import pandas as pd
from pprint import pprint

# ---------- helper ----------
def print_block(title):
    print("\n" + "-"*80)
    print(title)
    print("-"*80)

# doctor URIs we saw:
doc_uris = [
    "http://data.europa.eu/esco/occupation/71f09c8b-a172-408c-b9e7-32e580e39ff6",  # general practitioner
    "http://data.europa.eu/esco/occupation/9b889f07-c39c-464d-b9d9-b2daa650f9ac",  # specialised doctor
]

# digital URIs found earlier (you printed them) -- if not present, build from df_dig:
if 'digital_uris' in globals():
    dig_uris = [u for u in digital_uris if u in occ_skills.columns]
elif 'df_dig' in globals():
    dig_uris = df_dig['conceptUri'].tolist()
else:
    dig_uris = []

print_block("Doctor URIs and labels")
for u in doc_uris:
    lbl = None
    if hasattr(esco, 'occupations') and 'preferredLabel' in esco.occupations.columns:
        try:
            lbl = esco.occupations.set_index('conceptUri').loc[u, 'preferredLabel']
        except Exception:
            lbl = None
    print(u, "->", lbl)

# 1) If esco object contains occupation-skill relations (try a few likely attribute names)
print_block("Looking for occupation-skill relations inside esco object (common attr names: occupation_skill, occupation_skills, occ_skill, relations)")
found = False
for attr in ['occupation_skill', 'occupation_skills', 'occupation_skill_relations', 'occ_skill', 'occ_skills', 'occupationToSkills', 'occ2skill']:
    if hasattr(esco, attr):
        df_rel = getattr(esco, attr)
        print(f"Found esco.{attr} (type {type(df_rel)}). Showing head:")
        try:
            display(df_rel.head())
        except Exception:
            print(df_rel.head())
        found = True

if not found:
    print("No obvious relation attribute found inside esco object. That's fine — we try CSV fallback next.")

# 2) Try to load ESCO 'occupationSkillRelation' CSV if present in data_raw folder
raw_dir = os.path.join(useful_paths.data_raw, "esco", "v1.1.0")
candidate_files = [
    "occupationSkillRelation_en.csv",
    "occupations_skills.csv",
    "occupationSkill_en.csv",
    "occupationSkillRelation.csv",
    "skillRelation_en.csv"
]
print_block("Trying to find raw occupation-skill CSVs in data_raw/esco/v1.1.0")
for fn in candidate_files:
    p = os.path.join(raw_dir, fn)
    if os.path.exists(p):
        print("Found", p, " — reading (first rows):")
        try:
            df_rel = pd.read_csv(p, nrows=10)
            display(df_rel.head(10))
        except Exception as e:
            print("Failed to read", p, ":", e)

# 3) Direct check inside occupationsCollection_en.csv to see if any skill lists appear as columns
occs_csv = os.path.join(raw_dir, "occupationsCollection_en.csv")
if os.path.exists(occs_csv):
    print_block("Reading occupationsCollection_en.csv head")
    df_occs_raw = pd.read_csv(occs_csv, nrows=10)
    display(df_occs_raw.head(5))
    # check columns presence
    print("columns:", df_occs_raw.columns.tolist())
else:
    print("occupationsCollection_en.csv not found at", occs_csv)

# 4) Direct check: for each doctor URI, print how occ_skills marks the digital URIs (essential/optional/0)
print_block("Direct check in occ_skills matrix: doctor occupation x digital skills (value codes)")
for uri in doc_uris:
    if uri not in occ_skills.index:
        print(uri, "NOT in occ_skills index")
        continue
    subset = occ_skills.loc[uri, [c for c in dig_uris if c in occ_skills.columns]]
    # show non-zero entries only
    nonzero = subset[subset != 0]
    print("\nOccupation:", uri)
    if nonzero.empty:
        print(" -> No digital skill marked non-zero for this occupation in occ_skills.")
    else:
        print(" -> non-zero digital marks (skill_uri -> value):")
        for s,v in nonzero.items():
            print("   ", v, s)



--------------------------------------------------------------------------------
Doctor URIs and labels
--------------------------------------------------------------------------------
http://data.europa.eu/esco/occupation/71f09c8b-a172-408c-b9e7-32e580e39ff6 -> general practitioner
http://data.europa.eu/esco/occupation/9b889f07-c39c-464d-b9d9-b2daa650f9ac -> specialised doctor

--------------------------------------------------------------------------------
Looking for occupation-skill relations inside esco object (common attr names: occupation_skill, occupation_skills, occ_skill, relations)
--------------------------------------------------------------------------------
No obvious relation attribute found inside esco object. That's fine — we try CSV fallback next.

--------------------------------------------------------------------------------
Trying to find raw occupation-skill CSVs in data_raw/esco/v1.1.0
---------------------------------------------------------------------------

Final code for table in paper: Move to other notebook!

In [29]:
# Paste this at the BOTTOM of your notebook (after useful_paths is available)
import os
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

from data.framework import Esco

# Paths (exact filenames we used previously)
p_occ_skills = Path(useful_paths.data_processed) / "esco" / "occ_skills_matrix.pkl"
p_dig = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "digCompSkillsCollection_en.csv"
p_grn = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "greenSkillsCollection_en.csv"
p_occs = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "occupationsCollection_en.csv"
p_isco = Path(useful_paths.data_raw) / "esco" / "v1.1.0" / "iscoGroupsCollection_en.csv"  # optional

print("Loading occ-skills from:", p_occ_skills)
with open(p_occ_skills, "rb") as f:
    occ_skills = pickle.load(f)   # DataFrame indexed by occupation conceptUri, columns = skill URIs

print("occ_skills shape:", occ_skills.shape)
unique_vals = np.unique(occ_skills.values)
print("Unique values sample:", unique_vals[:10])

# detect essential / optional codes
nonzero = unique_vals[unique_vals != 0]
if len(nonzero) == 0:
    raise RuntimeError("occ_skills seems empty")
if set(nonzero) == {1, 2}:
    essential_code = 1
    optional_code = 2
else:
    # fallback: assume max is essential
    essential_code = int(nonzero.max())
    optional_code = int(min(nonzero))
print("Assumed codes -> essential_code:", essential_code, " optional_code:", optional_code)

# load skill lists
def load_skill_uris(path):
    if not path.exists():
        print(f"Skill list not found: {path}")
        return None, None
    df = pd.read_csv(path, index_col=0)
    if "conceptUri" in df.columns:
        uris = df["conceptUri"].astype(str).tolist()
    else:
        uris = df.index.astype(str).tolist()
    return uris, df

dig_uris, df_dig = load_skill_uris(p_dig)
grn_uris, df_grn = load_skill_uris(p_grn)
print("Found digital_uris:", len(dig_uris) if dig_uris is not None else None)
print("Found green_uris:", len(grn_uris) if grn_uris is not None else None)

# load occ metadata (prefer CSV, else use Esco())
esco = Esco()
if p_occs.exists():
    df_occs = pd.read_csv(p_occs, index_col=0)
    print("Loaded occupations CSV:", p_occs)
else:
    df_occs = esco.occupations.copy()
    print("Used esco.occupations fallback")

# ensure conceptUri index and columns
if "conceptUri" in df_occs.columns:
    df_occs = df_occs.set_index("conceptUri", drop=False)
else:
    if df_occs.index.name is None or "conceptUri" not in df_occs.index.name:
        df_occs = df_occs.reset_index().rename(columns={df_occs.index.name or "index": "conceptUri"}).set_index("conceptUri", drop=False)

# find ISCO column
possible_isco_cols = [c for c in df_occs.columns if "isco" in c.lower() or c.lower()=="code"]
if not possible_isco_cols:
    raise KeyError("No ISCO column found in occupations metadata. Columns: " + ", ".join(df_occs.columns))
isco_col = possible_isco_cols[0]
print("Using ISCO column:", isco_col)

# normalize to ISCO-3 code (take first 3 digits)
def to_isco3(val):
    if pd.isna(val):
        return None
    s = str(val)
    # remove non-digits
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) >= 3:
        return digits[:3]
    if len(digits)>0:
        return digits.zfill(3)[:3]
    return None

df_occs["isco3"] = df_occs[isco_col].apply(to_isco3)
# map occupations in occ_skills to isco3
occ_index = occ_skills.index.astype(str)
isco3_per_occ = occ_index.to_series().map(lambda u: df_occs.loc[u, "isco3"] if u in df_occs.index else None).astype("object")

# build boolean essential matrix and aggregate to ISCO-3
ess_bool = (occ_skills == essential_code)
# grouped_any: for each ISCO-3 row, whether any ISCO-4 occupation in that ISCO-3 has that skill essential
grouped_any = ess_bool.groupby(isco3_per_occ, axis=0).any()
skill_isco3_counts = grouped_any.sum(axis=0)  # for each skill: number of ISCO-3 groups where it's essential

# summary (a)
def summarize(skill_uris):
    if skill_uris is None:
        return None, None, None
    present = [u for u in skill_uris if u in skill_isco3_counts.index]
    n_skills = len(skill_uris)
    ever_essential_isco3 = int((skill_isco3_counts.reindex(present).fillna(0) > 0).sum()) if present else 0
    median_isco3_groups_per_skill = float(skill_isco3_counts.reindex(present).fillna(0).median()) if present else 0.0
    return n_skills, ever_essential_isco3, median_isco3_groups_per_skill

dig_n, dig_ever, dig_med = summarize(dig_uris)
grn_n, grn_ever, grn_med = summarize(grn_uris)

df_a = pd.DataFrame({
    "# skills": [dig_n, grn_n],
    "# ever essential (ISCO-3)": [dig_ever, grn_ever],
    "median # occs": [int(dig_med), int(grn_med)]
}, index=["Digital", "Green"])

print("\n=== (a) ISCO-3 aggregated coverage ===")
print(df_a.to_string())

# (b) per-ISCO3 rows: distinct essential counts and number of ISCO-4 occs
isco3_index = [c for c in grouped_any.index if c is not None]
rows = []
for code in isco3_index:
    n_green_distinct = int(grouped_any.loc[code, grn_uris].sum()) if grn_uris is not None else 0
    n_dig_distinct = int(grouped_any.loc[code, dig_uris].sum()) if dig_uris is not None else 0
    n_isco4 = int((isco3_per_occ == code).sum())
    label = None
    rows.append((code, label, n_green_distinct, n_dig_distinct, n_isco4))
df_isco3 = pd.DataFrame(rows, columns=["isco3","label","n_green_distinct","n_dig_distinct","n_isco4_occs"]).set_index("isco3")

# attach labels from isco CSV if available, else try esco.isco_groups
if p_isco.exists():
    df_isco_labels = pd.read_csv(p_isco, index_col=0)
    if "code" in df_isco_labels.columns:
        df_isco_labels = df_isco_labels.set_index("code")
    if "preferredLabel" in df_isco_labels.columns:
        label_map = df_isco_labels["preferredLabel"].to_dict()
        df_isco3["label"] = df_isco3.index.map(lambda c: label_map.get(str(c), f"ISCO-3 {c}"))
else:
    try:
        if hasattr(esco, "isco_groups"):
            df_isoof = esco.isco_groups.copy()
            if "code" in df_isoof.columns and "preferredLabel" in df_isoof.columns:
                df_isoof = df_isoof.set_index("code")
                label_map = df_isoof["preferredLabel"].to_dict()
                df_isco3["label"] = df_isco3.index.map(lambda c: label_map.get(str(c), f"ISCO-3 {c}"))
    except Exception:
        df_isco3["label"] = df_isco3.index.map(lambda c: f"ISCO-3 {c}")

# reorder columns for pretty printing
df_isco3 = df_isco3[["label","n_green_distinct","n_dig_distinct","n_isco4_occs"]]

top5_green = df_isco3.sort_values("n_green_distinct", ascending=False).head(5)
top5_dig = df_isco3.sort_values("n_dig_distinct", ascending=False).head(5)

print("\n=== (b1) Top 5 ISCO-3 by DISTINCT essential GREEN skills ===")
print(top5_green.to_string())

print("\n=== (b2) Top 5 ISCO-3 by DISTINCT essential DIGITAL skills ===")
print(top5_dig.to_string())

# Latex formatted two-subtable (ISCO-3 only) — printed so you can copy-paste
print("\n--- LaTeX table (ISCO-3 only, ready to paste) ---\n")
print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Essential-skill coverage aggregated to ISCO-3 groups (ESCO lists)}")
print(r"\label{tab:essential_skill_coverage_isco3}")
print(r"\begin{subtable}[t]{.48\textwidth}")
print(r"  \centering")
print(r"  \caption{(a) ISCO-3 aggregated skill coverage}")
print(r"  \begin{tabular}{lrrr}")
print(r"    \toprule")
print(r"      & \# skills & \# ever essential (ISCO-3) & median \# occs \\")
print(r"    \midrule")
print(f"    Digital & {dig_n} & {dig_ever} & {int(dig_med)} \\\\")
print(f"    Green   & {grn_n} & {grn_ever} & {int(grn_med)} \\\\")
print(r"    \bottomrule")
print(r"  \end{tabular}")
print(r"\end{subtable}%")
print(r"\hfill")
print(r"\begin{subtable}[t]{.48\textwidth}")
print(r"  \centering")
print(r"  \caption{(b) Top ISCO-3 groups by essential-skill counts}")
print(r"  \begin{tabular}{lrrr}")
print(r"    \toprule")
print(r"    Occupation (ISCO-3) & \# green skills & \# digital skills & \# ISCO-4 occs \\")
print(r"    \midrule")
for _, row in top5_green.iterrows():
    print(f"    {row['label']} ({row.name}) & {row['n_green_distinct']} & {row['n_dig_distinct']} & {row['n_isco4_occs']} \\\\")
print(r"    \midrule")
for _, row in top5_dig.iterrows():
    print(f"    {row['label']} ({row.name}) & {row['n_green_distinct']} & {row['n_dig_distinct']} & {row['n_isco4_occs']} \\\\")
print(r"    \bottomrule")
print(r"  \end{tabular}")
print(r"\end{subtable}")
print(r"\end{table}")


Loading occ-skills from: /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/processed/esco/occ_skills_matrix.pkl
occ_skills shape: (3008, 13891)


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/333771409.py:19: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  occ_skills = pickle.load(f)   # DataFrame indexed by occupation conceptUri, columns = skill URIs


Unique values sample: [0 1 2]
Assumed codes -> essential_code: 1  optional_code: 2
Found digital_uris: 21
Found green_uris: 570
Used esco.occupations fallback
Using ISCO column: iscoGroup

=== (a) ISCO-3 aggregated coverage ===
         # skills  # ever essential (ISCO-3)  median # occs
Digital        21                          5              0
Green         570                        475              1

=== (b1) Top 5 ISCO-3 by DISTINCT essential GREEN skills ===
                                                         label  n_green_distinct  n_dig_distinct  n_isco4_occs
isco3                                                                                                         
214    Engineering professionals (excluding electrotechnology)               177               0           125
213                                 Life science professionals               140               1            40
311               Physical and engineering science technicians               104      

/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_23018/333771409.py:99: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  grouped_any = ess_bool.groupby(isco3_per_occ, axis=0).any()


Inspection of model output for debugging coeffy and earnings